# All-Species CCF Validation — DH Tau B (De Regt+2024 §4.2)

Clean notebook covering three retrievals with the unified `run_species_ccf_validation`
function from `analysis.py`.  ACF is normalised at **rv = 0 km/s** (planet frame);
SNR = CCF(rv=0) / std(CCF − ACF_aligned) over the full ±1000 km/s grid.

| Retrieval | Guidebook | Likelihood | Cov | N_live | lnZ | χ² |
|-----------|-----------|------------|-----|--------|-----|----|
| 1918539 | v4.0 | Ruffio (buggy, no GP) | diag | 800 | 1,689,737 | 4.6 / 4.2 |
| 368546  | v4.0 | Ruffio (buggy, no GP) | diag | 800 | 1,689,733 | — |
| **3062330** | **v4.2** | **Std Gaussian (fixed)** | **GP** | **600** | **1,709,579** | **1.04 / 0.97** |

**Key fix (Guidebook v4.2)**: replaced Ruffio Eq. 17 with the correct Standard Gaussian
log-likelihood for `scale_flux in (False, 'physical')` mode. The GP run 3062330 shows
`log_a = 0.266` (not at prior boundary), confirming the GP is physically meaningful.


## 1. Imports & Common Setup

In [1]:
import sys
import pickle
import importlib.util
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import astropy.io.fits as fits
from pathlib import Path

matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 120})

RECIPE_DIR  = Path('/data2/peng/Recipe_DH_Tau_B')
WORKPATH    = Path('/data2/peng')
RETRIEVAL_BASE = WORKPATH / 'retrievals'

# analysis.py contains run_species_ccf_validation and helpers
sys.path.insert(0, str(RECIPE_DIR))
import analysis
from analysis import run_species_ccf_validation
importlib.reload(analysis)  # pick up latest edits
from analysis import run_species_ccf_validation

print('analysis.py loaded, run_species_ccf_validation ready')


analysis.py loaded, run_species_ccf_validation ready


## 2. Import Guidebook v4.0 (used for 1918539 and 368546)

In [2]:
_gb40_path = str(RECIPE_DIR / 'Tasting_guidebook' / 'Guidebook_GAStronomy_Piette_v4.2.py')
_spec = importlib.util.spec_from_file_location('Guidebook_v4_2', _gb40_path)
_gb40 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_gb40)

Target40                       = _gb40.Target
Parameters40                   = _gb40.Parameters
Retrieval40                    = _gb40.Retrieval
_load_night40                  = _gb40._load_night
make_free_params_equilibrium40 = _gb40.make_free_params_equilibrium
pRT_spectrum40                 = _gb40.pRT_spectrum

print('Guidebook v4.2 loaded')


Input data path changed to '/net/lem/data2/pRT3_formatted/input_data'
Guidebook v4.2 loaded


## 3. Load Observation Data

`_load_night` returns 1-D flattened arrays (needed for Target/Retrieval setup).  
`_load_3d` returns (nDet=3, nOrder=5, nPix=2048) cubes with NaN masking (needed for CCF validation).

In [3]:
# 1-D arrays for Target/Retrieval construction
wave_N1, flux_N1, err_N1, R_N1 = _load_night40(
    '2022-12-31',
    flux_file        = 'extracted_spectra_combined_sigmaclipper.npy',
    err_file         = 'extracted_spectra_combined_err_sigmaclipper.npy',
    normalize_method = None,
)
wave_N2, flux_N2, err_N2, R_N2 = _load_night40(
    '2023-01-01',
    flux_file        = 'extracted_spectra_combined_sigmaclipper_0101.npy',
    err_file         = 'extracted_spectra_combined_err_sigmaclipper_0101.npy',
    normalize_method = None,
)

# Target objects (1-D flat; used by Retrieval)
T1 = Target40(wl=wave_N1, fl=flux_N1, err=err_N1, name='dh_tau_b_N1')
T2 = Target40(wl=wave_N2, fl=flux_N2, err=err_N2, name='dh_tau_b_N2')
print(f'N1: {len(wave_N1)} valid px   R_N1 = {R_N1:.0f}')
print(f'N2: {len(wave_N2)} valid px   R_N2 = {R_N2:.0f}')


def _load_3d(night_str):
    """Load (nDet=3, nOrder=5, nPix=2048) cubes with NaN for bad pixels.

    Required by run_species_ccf_validation (expects 3-D arrays, not 1-D flat).
    """
    base = Path(f'/data2/peng/{night_str}')
    flux = np.load(base / 'extracted_spectra_combined_sigmaclipper.npy').astype(float)   # (3,5,2048)
    err  = np.load(base / 'extracted_spectra_combined_err_sigmaclipper.npy').astype(float)
    hdu  = fits.open(base / 'cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')
    wave = np.array(hdu[1].data, dtype=float)[:, :5, :]                              # (3,5,2048)
    bad  = ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~np.isfinite(wave)
    flux[bad] = np.nan
    err[bad]  = np.nan
    wave[bad] = np.nan
    print(f'  [{night_str}] 3-D shape: {flux.shape}  valid pix: {(~bad).sum()}')
    return wave, flux, err


wave3_N1, flux3_N1, err3_N1 = _load_3d('2022-12-31')
wave3_N2, flux3_N2, err3_N2 = _load_3d('2023-01-01')

# Model wavelength grid — same for all retrievals (identical instrument setup).
# The raw .npy file is ordered by (order, det, pix) with wavelength DECREASING across
# order boundaries (4 backward jumps), which breaks np.interp.  Sort ascending here
# so every downstream np.interp call works correctly.
wave_model = np.load(
    RETRIEVAL_BASE / '2968924_N600_ev0.5_Normper_chip_median_PerChipScaleFalse/retrieval_model_wave.npy'
)
wave_model = np.sort(wave_model)
print(f'wave_model: {len(wave_model)} pts  [{wave_model[0]:.1f}, {wave_model[-1]:.1f}] nm  '
      f'(sorted ascending)')


  [2022-12-31] No normalisation applied.
  [2022-12-31] Estimating resolving power...
  Estimated R = 314490 (median over 15 chips, range 297831–333720)
  [2022-12-31] Valid pixels: 26360 / 30720
  [2023-01-01] No normalisation applied.
  [2023-01-01] Estimating resolving power...
  Estimated R = 314480 (median over 15 chips, range 297860–333843)
  [2023-01-01] Valid pixels: 26915 / 30720
[Target dh_tau_b_N1] wl:(26360,)  fl:(26360,)  err:(26360,)
[Target dh_tau_b_N2] wl:(26915,)  fl:(26915,)  err:(26915,)
N1: 26360 valid px   R_N1 = 314490
N2: 26915 valid px   R_N2 = 314480
  [2022-12-31] 3-D shape: (3, 5, 2048)  valid pix: 26360
  [2023-01-01] 3-D shape: (3, 5, 2048)  valid pix: 24964
wave_model: 26360 pts  [2063.9, 2471.4] nm  (sorted ascending)


## 4. TRACE_SPECIES Definitions

In [4]:
# Base species — equilibrium retrieval without Ti
TRACE_SPECIES_BASE = {
    '1H2-16O':     'H2O',
    '12C-16O':     '12CO',
    '13C-16O':     '13CO',
    '12C-1H4__MM': 'CH4',
    '56Fe-1H':     'FeH',
    '1H-19F':      'HF',
    '23Na':        'Na',
    '40Ca':        'Ca',
}

# With Ti — for retrieval 368546 (run with Ti uncommented in v4.0)
# Note: Ti will be skipped if not present in mass_fractions (e.g., if the
# v4.0 Guidebook was re-loaded with Ti commented out in EQ_SPECIES_PRT3).
TRACE_SPECIES_TI = dict(TRACE_SPECIES_BASE)
TRACE_SPECIES_TI['48Ti'] = 'Ti'

print('TRACE_SPECIES_BASE:', list(TRACE_SPECIES_BASE.values()))
print('TRACE_SPECIES_TI  :', list(TRACE_SPECIES_TI.values()))


TRACE_SPECIES_BASE: ['H2O', '12CO', '13CO', 'CH4', 'FeH', 'HF', 'Na', 'Ca']
TRACE_SPECIES_TI  : ['H2O', '12CO', '13CO', 'CH4', 'FeH', 'HF', 'Na', 'Ca', 'Ti']


---
## §5 — Retrieval 1918539
**Guidebook v4.0 · diagonal covariance · Ruffio likelihood (buggy) · N_live = 800**  
Reference: cloud-free equilibrium run, no Ti, no GP.


In [5]:
RID_A  = '1735916_N600_ev0.5_NormNone_PerChipScaleFalse'
DIR_A  = RETRIEVAL_BASE / RID_A
LABEL_A = '1735916'

with open(DIR_A / 'final_params_dict.pickle', 'rb') as f:
    best_fit_A = pickle.load(f)
print('Best-fit params (1735916):')
for k, v in best_fit_A.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

constant_params_A = {'chemistry': 'equilibrium'}
free_params_A     = make_free_params_equilibrium40()
parameters_A      = Parameters40(free_params_A, constant_params_A)
parameters_A(np.random.rand(parameters_A.ndim))
parameters_A.params.update(best_fit_A)

retrieval_A = Retrieval40(
    parameters         = parameters_A,
    N_live_points      = 800,
    evidence_tolerance = 0.5,
    targets            = [T1, T2],
    testing            = False,
    normalize_flux     = None,
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = True,
)
retrieval_A.parameters.params.update(best_fit_A)
print('Retrieval 1735916 ready')


Best-fit params (1735916):
  rv_N1                     = 31.607
  rv_N2                     = 31.568
  vsini                     = 7.5025
  epsilon                   = 0.77168
  log_M                     = 1.1888
  log_R                     = 0.48153
  T_anchor                  = 2107.6
  dT_1                      = 686.81
  dT_2                      = 429.43
  dT_3                      = 51.022
  dT_4                      = 151.86
  dT_5                      = 245.37
  dT_6                      = 155.95
  dT_7                      = 293.38
  C_H                       = 0.056289
  C/O                       = 0.65947
  log_12CO_13CO             = 1.7747
  F_H                       = -0.49457
  log_Na                    = -4.12
  log_Ca                    = -4.9128
  log_X_MgSiO3              = -0.72687
  log_X_Fe                  = -1.1064
  fsed                      = 2.9376
  log_Kzz                   = 10.966
  sigma_lnorm               = 1.8413
  log_a                     = 0.26565


Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


...

 Loading line opacities of species '12C-16O2__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO2/12C-16O2/12C-16O2__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 1735916 ready


In [6]:
results_A = run_species_ccf_validation(
    retrieval          = retrieval_A,
    pRT_spectrum_class = pRT_spectrum40,
    best_fit_params    = best_fit_A,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_N1,
    err_N1             = err3_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_N2,
    err_N2             = err3_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_A,
    retrieval_label    = LABEL_A,
    use_absolute_flux  = True,
)


  Planet RV: N1 = +31.607 km/s  N2 = +31.568 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [2.83e-16, 1.26e-15]
  Generating no-X templates...


    [H2O] template RMS = 2.22e-16
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.44e-17


    [13CO] template RMS = 8.05e-18


...

      peak CCF = 1.180e-12   peak_rv = -835.0 km/s   SNR = 3.47   template_fraction = 7.6e-15  [template below sensitivity]
    [HF]...


      peak CCF = 2.270e-11   peak_rv = -464.0 km/s   SNR = 2.74   template_fraction = 8.3e-14  [template below sensitivity]
    [Na]...


      peak CCF = 5.591e-11   peak_rv = -895.0 km/s   SNR = 2.73   template_fraction = 1.3e-13  [template below sensitivity]
    [Ca]...


      peak CCF = 1.970e-11   peak_rv = +997.0 km/s   SNR = 2.34   template_fraction = 2.3e-13  [template below sensitivity]
    [totalCO]...


      peak CCF = 9.819e-10   peak_rv = +0.0 km/s   SNR = 8.68   template_fraction = 5.6e-13  [template below sensitivity]
  Saving per-species CCF panels...
    [H2O] template below sensitivity — skipping panel
    [12CO] template below sensitivity — skipping panel
    [13CO] template below sensitivity — skipping panel
    [CH4] template below sensitivity — skipping panel
    [FeH] template below sensitivity — skipping panel
    [HF] template below sensitivity — skipping panel
    [Na] template below sensitivity — skipping panel
    [Ca] template below sensitivity — skipping panel
    [totalCO] template below sensitivity — skipping panel
  [H2O] excluded from SNR plot: template_fraction = 9.2e-13 < 1e-04
  [12CO] excluded from SNR plot: template_fraction = 5.5e-13 < 1e-04
  [13CO] excluded from SNR plot: template_fraction = 2.9e-13 < 1e-04
  [CH4] excluded from SNR plot: template_fraction = 6.0e-17 < 1e-04
  [FeH] excluded from SNR plot: template_fraction = 7.6e-15 < 1e-04
  [HF] exclu

---
## §6 — Retrieval 368546
**Guidebook v4.0 · diagonal covariance · Ruffio likelihood (buggy) · N_live = 800 · +Ti**  
First [F/H] parameterisation run; includes Ti as a free parameter.  
Note: Ti CCF will be skipped if Ti is absent from the v4.0 mass_fractions (Ti was
uncommented in the original run script but is commented out in the Guidebook file as saved).


In [7]:
RID_B  = '368546_N800_ev0.5_NormNone_PerChipScaleFalse'
DIR_B  = RETRIEVAL_BASE / RID_B
LABEL_B = '368546'

with open(DIR_B / 'final_params_dict.pickle', 'rb') as f:
    best_fit_B = pickle.load(f)
print('Best-fit params (368546):')
for k, v in best_fit_B.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

constant_params_B = {'chemistry': 'equilibrium'}
free_params_B     = make_free_params_equilibrium40()
# 368546 was run with Ti as a free parameter — inject it into free_params
free_params_B['log_Ti'] = ([-12, -2], r'$\log$ Ti')
parameters_B      = Parameters40(free_params_B, constant_params_B)
parameters_B(np.random.rand(parameters_B.ndim))
parameters_B.params.update(best_fit_B)

retrieval_B = Retrieval40(
    parameters         = parameters_B,
    N_live_points      = 800,
    evidence_tolerance = 0.5,
    targets            = [T1, T2],
    testing            = False,
    normalize_flux     = None,
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = True,
)
retrieval_B.parameters.params.update(best_fit_B)
print('Retrieval 368546 ready')


Best-fit params (368546):
  rv_N1                     = 31.608
  rv_N2                     = 31.541
  vsini                     = 7.4374
  epsilon                   = 0.73311
  log_M                     = 1.105
  log_R                     = 0.4629
  T_anchor                  = 2185
  dT_1                      = 601.33
  dT_2                      = 401.65
  dT_3                      = 96.443
  dT_4                      = 137.22
  dT_5                      = 294.89
  dT_6                      = 268.79
  dT_7                      = 438.37
  C_H                       = -0.48011
  C/O                       = 0.58858
  log_12CO_13CO             = 1.3025
  F_H                       = -0.58744
  log_Na                    = -5.7391
  log_Ca                    = -4.9497
  log_Ti                    = -9.0823
  [C/H]                     = -0.43769
  [C/H]_xsolar              = 0.36501
  s2                        = 1
  chi2                      = 4.5941
  chi2_N2                   = 4.1959
  lnZ   

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '14N-1H3' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/NH3/14N-1H3/14N-1H3__CoYuTe.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 368546 ready


In [8]:
results_B = run_species_ccf_validation(
    retrieval          = retrieval_B,
    pRT_spectrum_class = pRT_spectrum40,
    best_fit_params    = best_fit_B,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_N1,
    err_N1             = err3_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_N2,
    err_N2             = err3_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_TI,
    retrieval_dir      = DIR_B,
    retrieval_label    = LABEL_B,
    use_absolute_flux  = True,
)


  Planet RV: N1 = +31.608 km/s  N2 = +31.541 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [2.67e-16, 1.41e-15]
  Generating no-X templates...


    [H2O] template RMS = 1.92e-16
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.51e-17


    [13CO] template RMS = 1.15e-17


...

      peak CCF = 1.187e-12   peak_rv = -835.0 km/s   SNR = 3.73   template_fraction = 6.6e-15  [template below sensitivity]
    [HF]...


      peak CCF = 3.070e-11   peak_rv = -464.0 km/s   SNR = 2.78   template_fraction = 1.2e-13  [template below sensitivity]
    [Na]...


      peak CCF = 3.258e-11   peak_rv = -336.0 km/s   SNR = 3.40   template_fraction = 7.5e-14  [template below sensitivity]
    [Ca]...


      peak CCF = 3.429e-11   peak_rv = +333.0 km/s   SNR = 2.22   template_fraction = 4.0e-13  [template below sensitivity]
    [totalCO]...


      peak CCF = 1.141e-09   peak_rv = +0.0 km/s   SNR = 9.43   template_fraction = 5.4e-13  [template below sensitivity]
  Saving per-species CCF panels...
    [H2O] template below sensitivity — skipping panel
    [12CO] template below sensitivity — skipping panel
    [13CO] template below sensitivity — skipping panel
    [CH4] template below sensitivity — skipping panel
    [FeH] template below sensitivity — skipping panel
    [HF] template below sensitivity — skipping panel
    [Na] template below sensitivity — skipping panel
    [Ca] template below sensitivity — skipping panel
    [totalCO] template below sensitivity — skipping panel
  [H2O] excluded from SNR plot: template_fraction = 8.1e-13 < 1e-04
  [12CO] excluded from SNR plot: template_fraction = 5.3e-13 < 1e-04
  [13CO] excluded from SNR plot: template_fraction = 4.1e-13 < 1e-04
  [CH4] excluded from SNR plot: template_fraction = 2.9e-17 < 1e-04
  [FeH] excluded from SNR plot: template_fraction = 6.6e-15 < 1e-04
  [HF] exclu

---
## §7 — Retrieval 3062330
**Guidebook v4.2 · GP covariance · Standard Gaussian likelihood (fixed) · N_live = 600**  
First GP run with the correct likelihood formula.  `log_a = 0.266` (not at boundary) confirms
the GP is physically meaningful.  `χ² ≈ 1.04` confirms the model fits the data well.

| Parameter | MAP value | σ |
|-----------|-----------|---|
| log_a     | 0.266  (a = 1.84) | 0.00116 |
| log_l     | −2.284 (l = 5.2 nm) | 0.00146 |
| lnZ       | 1,709,579 | — |
| χ²        | 1.04 / 0.97 | — |


In [5]:
# Import Guidebook v4.2 for GP-capable pRT_spectrum
_gb42_path = str(RECIPE_DIR / 'Tasting_guidebook' / 'Guidebook_GAStronomy_Piette_v4.2.py')
_spec42 = importlib.util.spec_from_file_location('Guidebook_v4_2', _gb42_path)
_gb42   = importlib.util.module_from_spec(_spec42)
_spec42.loader.exec_module(_gb42)

Target42                       = _gb42.Target
Parameters42                   = _gb42.Parameters
Retrieval42                    = _gb42.Retrieval
make_free_params_equilibrium42 = _gb42.make_free_params_equilibrium
pRT_spectrum42                 = _gb42.pRT_spectrum

# Re-create Target objects from v4.2 (same data, just the class origin differs)
T1_42 = Target42(wl=wave_N1, fl=flux_N1, err=err_N1, name='dh_tau_b_N1')
T2_42 = Target42(wl=wave_N2, fl=flux_N2, err=err_N2, name='dh_tau_b_N2')

print('Guidebook v4.2 loaded')


Input data path changed to '/net/lem/data2/pRT3_formatted/input_data'
[Target dh_tau_b_N1] wl:(26360,)  fl:(26360,)  err:(26360,)
[Target dh_tau_b_N2] wl:(26915,)  fl:(26915,)  err:(26915,)
Guidebook v4.2 loaded


In [10]:
RID_C  = '3062330_N600_ev0.5_NormNone_PerChipScaleFalse'
DIR_C  = RETRIEVAL_BASE / RID_C
LABEL_C = '3062330'

with open(DIR_C / 'final_params_dict.pickle', 'rb') as f:
    best_fit_C = pickle.load(f)
print('Best-fit params (3062330):')
for k, v in best_fit_C.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

constant_params_C = {'chemistry': 'equilibrium'}
free_params_C     = make_free_params_equilibrium42()
# Add GP free parameters (log_a, log_l) used in the actual run
free_params_C['log_a'] = ([-1.0,  1.0], r'$\log a$')
free_params_C['log_l'] = ([-3.0,  0.0], r'$\log l$')

parameters_C = Parameters42(free_params_C, constant_params_C)
parameters_C(np.random.rand(parameters_C.ndim))
parameters_C.params.update(best_fit_C)

retrieval_C = Retrieval42(
    parameters         = parameters_C,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_42, T2_42],
    testing            = False,
    normalize_flux     = None,
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = True,
    cov_mode           = 'GP',
)
retrieval_C.parameters.params.update(best_fit_C)
print('Retrieval 3062330 (GP, v4.2) ready')


Best-fit params (3062330):
  rv_N1                     = 31.624
  rv_N2                     = 31.566
  vsini                     = 7.479
  epsilon                   = 0.79977
  log_M                     = 1.1233
  log_R                     = 0.46273
  T_anchor                  = 2162.6
  dT_1                      = 656.26
  dT_2                      = 388.89
  dT_3                      = 114.84
  dT_4                      = 121
  dT_5                      = 295.87
  dT_6                      = 266.36
  dT_7                      = 511.25
  C_H                       = -0.47526
  C/O                       = 0.55499
  log_12CO_13CO             = 1.3922
  F_H                       = -0.67462
  log_Na                    = -5.7711
  log_Ca                    = -4.9356
  log_a                     = 0.2656
  log_l                     = -2.2837
  [C/H]                     = -0.50573
  [C/H]_xsolar              = 0.31208
  s2                        = 1
  chi2                      = 1.0365
  chi2_

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '14N-1H3' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/NH3/14N-1H3/14N-1H3__CoYuTe.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 3062330 (GP, v4.2) ready


In [11]:
results_C = run_species_ccf_validation(
    retrieval          = retrieval_C,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_C,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_N1,
    err_N1             = err3_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_N2,
    err_N2             = err3_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_C,
    retrieval_label    = LABEL_C,
    use_absolute_flux  = True,
)


  Planet RV: N1 = +31.624 km/s  N2 = +31.566 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [2.54e-16, 1.36e-15]
  Generating no-X templates...


    [H2O] template RMS = 2.01e-16
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.27e-17


    [13CO] template RMS = 9.79e-18


...

      peak CCF = 9.624e-13   peak_rv = -835.0 km/s   SNR = 3.67   template_fraction = 5.5e-15  [template below sensitivity]
    [HF]...


      peak CCF = 2.577e-11   peak_rv = -464.0 km/s   SNR = 2.82   template_fraction = 9.4e-14  [template below sensitivity]
    [Na]...


      peak CCF = 3.034e-11   peak_rv = -336.0 km/s   SNR = 3.76   template_fraction = 6.3e-14  [template below sensitivity]
    [Ca]...


      peak CCF = 3.063e-11   peak_rv = +333.0 km/s   SNR = 2.24   template_fraction = 3.6e-13  [template below sensitivity]
    [totalCO]...


      peak CCF = 1.090e-09   peak_rv = +0.0 km/s   SNR = 9.53   template_fraction = 5.2e-13  [template below sensitivity]
  Saving per-species CCF panels...
    [H2O] template below sensitivity — skipping panel
    [12CO] template below sensitivity — skipping panel
    [13CO] template below sensitivity — skipping panel
    [CH4] template below sensitivity — skipping panel
    [FeH] template below sensitivity — skipping panel
    [HF] template below sensitivity — skipping panel
    [Na] template below sensitivity — skipping panel
    [Ca] template below sensitivity — skipping panel
    [totalCO] template below sensitivity — skipping panel
  [H2O] excluded from SNR plot: template_fraction = 8.5e-13 < 1e-04
  [12CO] excluded from SNR plot: template_fraction = 5.1e-13 < 1e-04
  [13CO] excluded from SNR plot: template_fraction = 3.5e-13 < 1e-04
  [CH4] excluded from SNR plot: template_fraction = 2.4e-17 < 1e-04
  [FeH] excluded from SNR plot: template_fraction = 5.5e-15 < 1e-04
  [HF] exclu

---
## §8 — Retrieval 657632  (008PM-EQ-RUFFIO)
**Guidebook v4.2 · GP covariance · Ruffio φ per-chip marginal likelihood · N_live = 600**  
First run with correct two-night per-chip φ likelihood (bug fix: night-2 shape mismatch).
Data is per-chip-median normalised (~1.0); model in raw pRT units (~10⁷ W/m²/μm).

**CCF flux handling**: data is normalised to ~1.0 via per-chip median division (Picos+2024 §3.2).
Model templates are scaled to data units using `model_scale_factor ≈ median(φ)` computed from
the best-fit `retrieval_model_flux_scaled.npy / retrieval_model_flux.npy` ratio.

> **NOTE**: `final_params_dict.pickle` and retrieval output npy files are only available
> after the retrieval completes. Cells below will error until then.

> **Note (2026-07-06):** the embedded outputs of the run cell below were cleared — they came from the pre-bugfix 2026-07-04 execution (buggy `_pcm_3d` chip mapping). The **valid post-fix results** are the `validation_deregt_*.png` files of 2026-07-05 in the retrieval output directory (regenerated in a fresh kernel from the fixed notebook, without saving outputs back).

In [12]:
RID_D   = '657632_N600_ev0.5_NormNone_PerChipScaleTrue'
DIR_D   = RETRIEVAL_BASE / RID_D
LABEL_D = '657632'

# import_gb42 (§7) defines the *42 aliases from the same file as import_gb40.
# If §7 was skipped (fresh kernel running §8 directly), fall back to _gb40.
try:
    _ = make_free_params_equilibrium42
except NameError:
    Target42                       = _gb40.Target
    Parameters42                   = _gb40.Parameters
    Retrieval42                    = _gb40.Retrieval
    make_free_params_equilibrium42 = _gb40.make_free_params_equilibrium
    pRT_spectrum42                 = _gb40.pRT_spectrum
    print('  [fallback] *42 aliases set from _gb40 (same Guidebook v4.2 file)')

with open(DIR_D / 'final_params_dict.pickle', 'rb') as f:
    best_fit_D = pickle.load(f)
print('Best-fit params (657632):')
for k, v in best_fit_D.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

# K2166 chip wavelength boundaries (nm) — 5 orders × 3 detectors
_K2166_D = np.array([
    [[2063.711, 2077.942], [2078.967, 2092.559], [2093.479, 2106.392]],
    [[2143.087, 2157.855], [2158.914, 2173.020], [2173.983, 2187.386]],
    [[2228.786, 2244.133], [2245.229, 2259.888], [2260.904, 2274.835]],
    [[2321.596, 2337.568], [2338.704, 2353.961], [2355.035, 2369.534]],
    [[2422.415, 2439.061], [2440.243, 2456.145], [2457.275, 2472.388]],
])

def _pcm_1d(wave, flux, err, K2166):
    """Per-chip median normalisation for 1-D arrays (returns copies)."""
    flux = flux.copy(); err = err.copy()
    for order in range(5):
        for det in range(3):
            mask = (wave >= K2166[order, det, 0]) & (wave <= K2166[order, det, 1])
            if mask.sum() < 10: continue
            med = np.nanmedian(flux[mask])
            if med > 0: flux[mask] /= med; err[mask] /= med
    return flux, err

def _pcm_3d(wave3, flux3, err3, K2166):
    """Per-chip median normalisation for 3-D cubes (n_det, n_order, n_pix); returns copies.

    Each chip is normalised by the median of its own finite pixels (same as the
    Guidebook's _load_night). NOTE (bugfix 2026-07-04): an earlier version
    selected pixels via the box K2166[order, det], but the cubes' order axis is
    REVERSED relative to the K2166 table (order index 0 = reddest chip = K2166
    row 4), so 12/15 chips silently failed the box mask and were left
    un-normalised — this poisoned the CCF residuals with full-model structure
    and produced spurious rv=0 detections (e.g. CH4 SNR ~19-21 in Sec.8/9).
    Never map chip indices to K2166 rows positionally.
    """
    flux3 = flux3.copy(); err3 = err3.copy()
    for det in range(flux3.shape[0]):
        for order in range(flux3.shape[1]):
            fin = np.isfinite(flux3[det, order, :])
            if fin.sum() < 10:
                print(f'  [_pcm_3d] WARNING: det{det} ord{order}: only '
                      f'{fin.sum()} finite px — chip left un-normalised')
                continue
            med = np.nanmedian(flux3[det, order, fin])
            if med > 0: flux3[det, order, :] /= med; err3[det, order, :] /= med
    return flux3, err3

# Normalise 1-D data for Target construction
flux_D_N1, err_D_N1 = _pcm_1d(wave_N1, flux_N1, err_N1, _K2166_D)
flux_D_N2, err_D_N2 = _pcm_1d(wave_N2, flux_N2, err_N2, _K2166_D)

# Normalise 3-D cubes for CCF validation
flux3_D_N1, err3_D_N1 = _pcm_3d(wave3_N1, flux3_N1, err3_N1, _K2166_D)
flux3_D_N2, err3_D_N2 = _pcm_3d(wave3_N2, flux3_N2, err3_N2, _K2166_D)

print(f'flux_D_N1 median: {np.nanmedian(flux_D_N1):.4f}  (should be ~1.0)')
print(f'flux_D_N2 median: {np.nanmedian(flux_D_N2):.4f}')

# Build Retrieval with per-chip-norm data — matches retrieval 657632 setup exactly
constant_params_D = {'chemistry': 'equilibrium'}
free_params_D     = make_free_params_equilibrium42()
del free_params_D['log_M']
del free_params_D['log_R']
free_params_D['log_g'] = ({'type': 'gaussian', 'mu': 3.64, 'sigma': 0.20}, r'$\log g$')
free_params_D['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_D['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')

parameters_D = Parameters42(free_params_D, constant_params_D)
parameters_D(np.random.rand(parameters_D.ndim))
parameters_D.params.update(best_fit_D)

T1_D = Target42(wl=wave_N1, fl=flux_D_N1, err=err_D_N1, name='dh_tau_b_D_N1')
T2_D = Target42(wl=wave_N2, fl=flux_D_N2, err=err_D_N2, name='dh_tau_b_D_N2')

retrieval_D = Retrieval42(
    parameters         = parameters_D,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_D, T2_D],
    testing            = False,
    normalize_flux     = None,
    per_chip_scaling   = True,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = False,
    cov_mode           = 'GP',
)
retrieval_D.parameters.params.update(best_fit_D)
print('Retrieval 657632 (Ruffio, GP, v4.2) ready')

Best-fit params (657632):
  rv_N1                     = 31.487
  rv_N2                     = 31.572
  vsini                     = 6.9002
  epsilon                   = 0.70172
  T_anchor                  = 1954.6
  dT_1                      = 583.77
  dT_2                      = 107.12
  dT_3                      = 156.91
  dT_4                      = 92.97
  dT_5                      = 184.5
  dT_6                      = 244.25
  dT_7                      = 387.96
  C_H                       = -0.2492
  C/O                       = 0.51233
  log_12CO_13CO             = 1.707
  F_H                       = -0.60438
  log_Na                    = -4.2655
  log_Ca                    = -4.445
  log_g                     = 3.6977
  log_a                     = 0.075844
  log_l                     = -2.1971
  [C/H]                     = -0.27392
  [C/H]_xsolar              = 0.53221
  s2                        = 1.3693
  chi2                      = 1.8749
  chi2_N2                   = 1.7408
  l

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '14N-1H3' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/NH3/14N-1H3/14N-1H3__CoYuTe.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 657632 (Ruffio, GP, v4.2) ready


In [13]:
# Compute global scale factor from best-fit retrieved outputs.
# flux_raw    : raw pRT emission flux (~10^7 W/m²/μm)
# flux_scaled : best-fit φ-scaled model — in same units as per-chip-norm data (~1.0)
# model_scale_D ≈ median(φ) across all chips, used to bring CCF templates to data units.
flux_raw_D    = np.load(DIR_D / 'retrieval_model_flux.npy').flatten().astype(float)
flux_scaled_D = np.load(DIR_D / 'retrieval_model_flux_scaled.npy').flatten().astype(float)

valid_m = np.isfinite(flux_raw_D) & (flux_raw_D > 0) & np.isfinite(flux_scaled_D)
model_scale_D = (np.nanmedian(flux_scaled_D[valid_m])
                 / np.nanmedian(flux_raw_D[valid_m]))
print(f'model_scale_factor = {model_scale_D:.4e}')
print(f'  → raw pRT ({1/model_scale_D:.2e} W/m²/μm median) → data units (~1.0)')

model_scale_factor = 3.4313e-06
  → raw pRT (2.91e+05 W/m²/μm median) → data units (~1.0)


In [ ]:
results_D = run_species_ccf_validation(
    retrieval          = retrieval_D,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_D,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_D_N1,
    err_N1             = err3_D_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_D_N2,
    err_N2             = err3_D_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_D,
    retrieval_label    = LABEL_D,
    use_absolute_flux  = False,
    model_scale_factor = model_scale_D,
)

---
## §9 — Retrieval 2968924  (008PM-EQ-MEDNORM)
**Guidebook v4.2 · GP covariance · Standard Gaussian likelihood (φ = 1 fixed) · N_live = 600**
Per-chip median normalisation applied to BOTH data and model (new Guidebook `'per_chip_median'`
option, added 2026-07-04); sigmaclipper data (no absolute flux calibration), direct `log_g`
(no log_M/log_R) — sibling to §8 (657632, 008PM-EQ-RUFFIO), but the per-chip scale is fixed
by construction (shared median) instead of marginalised via a Ruffio φ.

**CCF flux handling**: unlike §8, no `model_scale_factor` is needed here — `pRT_spectrum`'s
own `_apply_normalization()` produces a per-chip-median-normalised model directly (matching
the ~1.0-normalised data) whenever `normalize_flux='per_chip_median'` is set on the Retrieval
object, so a freshly generated model for the CCF templates is already on the right scale.

| Parameter | MAP value |
|-----------|-----------|
| rv_N1     | 31.471 km/s |
| rv_N2     | 31.493 km/s |
| vsini     | 6.007 km/s |
| C/O       | 0.597 |
| log_g     | 3.618 |
| log_a     | 0.267 |
| log_l     | −2.289 |
| lnZ       | −18,064.9 (not comparable to absolute-flux lnZ values above — different data units) |
| χ²        | 1.042 / 0.959 |

> **Note (2026-07-06):** the embedded outputs of the run cell below were cleared — they came from the pre-bugfix 2026-07-04 execution (buggy `_pcm_3d` chip mapping). The **valid post-fix results** are the `validation_deregt_*.png` files of 2026-07-05 in the retrieval output directory (regenerated in a fresh kernel from the fixed notebook, without saving outputs back).

In [15]:
RID_E  = '2968924_N600_ev0.5_Normper_chip_median_PerChipScaleFalse'
DIR_E  = RETRIEVAL_BASE / RID_E
LABEL_E = '2968924'

# Reuse _gb42 / make_free_params_equilibrium42 / pRT_spectrum42 from §7.
# Fall back to _gb40 if §7 was skipped.
try:
    _ = make_free_params_equilibrium42
except NameError:
    Target42                       = _gb40.Target
    Parameters42                   = _gb40.Parameters
    Retrieval42                    = _gb40.Retrieval
    make_free_params_equilibrium42 = _gb40.make_free_params_equilibrium
    pRT_spectrum42                 = _gb40.pRT_spectrum
    print('  [fallback] *42 aliases set from _gb40 (same Guidebook v4.2 file)')

# Define the per-chip-median helpers + K2166 table UNCONDITIONALLY.
# (An earlier version guarded this behind `try: _ = _pcm_1d`, which silently
# reused a stale/buggy in-memory _pcm_3d in a warm kernel and masked the
# 2026-07-04 chip-index bugfix — always redefine instead.)
_K2166_D = np.array([
    [[2063.711, 2077.942], [2078.967, 2092.559], [2093.479, 2106.392]],
    [[2143.087, 2157.855], [2158.914, 2173.020], [2173.983, 2187.386]],
    [[2228.786, 2244.133], [2245.229, 2259.888], [2260.904, 2274.835]],
    [[2321.596, 2337.568], [2338.704, 2353.961], [2355.035, 2369.534]],
    [[2422.415, 2439.061], [2440.243, 2456.145], [2457.275, 2472.388]],
])

def _pcm_1d(wave, flux, err, K2166):
    """Per-chip median normalisation for 1-D arrays (returns copies)."""
    flux = flux.copy(); err = err.copy()
    for order in range(5):
        for det in range(3):
            mask = (wave >= K2166[order, det, 0]) & (wave <= K2166[order, det, 1])
            if mask.sum() < 10: continue
            med = np.nanmedian(flux[mask])
            if med > 0: flux[mask] /= med; err[mask] /= med
    return flux, err

def _pcm_3d(wave3, flux3, err3, K2166):
    """Per-chip median normalisation for 3-D cubes (n_det, n_order, n_pix); returns copies.

    Each chip is normalised by the median of its own finite pixels (same as the
    Guidebook's _load_night). NOTE (bugfix 2026-07-04): an earlier version
    selected pixels via the box K2166[order, det], but the cubes' order axis is
    REVERSED relative to the K2166 table (order index 0 = reddest chip = K2166
    row 4), so 12/15 chips silently failed the box mask and were left
    un-normalised — this poisoned the CCF residuals with full-model structure
    and produced spurious rv=0 detections (e.g. CH4 SNR ~19-21 in Sec.8/9).
    Never map chip indices to K2166 rows positionally.
    """
    flux3 = flux3.copy(); err3 = err3.copy()
    for det in range(flux3.shape[0]):
        for order in range(flux3.shape[1]):
            fin = np.isfinite(flux3[det, order, :])
            if fin.sum() < 10:
                print(f'  [_pcm_3d] WARNING: det{det} ord{order}: only '
                      f'{fin.sum()} finite px — chip left un-normalised')
                continue
            med = np.nanmedian(flux3[det, order, fin])
            if med > 0: flux3[det, order, :] /= med; err3[det, order, :] /= med
    return flux3, err3

with open(DIR_E / 'final_params_dict.pickle', 'rb') as f:
    best_fit_E = pickle.load(f)
print('Best-fit params (2968924):')
for k, v in best_fit_E.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

# 008PM-EQ-MEDNORM free params: equilibrium chem, direct log_g (no log_M/log_R),
# GP noise — matches tasting_retrieval_equa_chem_v6.1_piette_cloudfree_mednorm.py exactly.
constant_params_E = {'chemistry': 'equilibrium'}
free_params_E     = make_free_params_equilibrium42()
del free_params_E['log_M']
del free_params_E['log_R']
free_params_E['log_g'] = ({'type': 'gaussian', 'mu': 3.64, 'sigma': 0.20}, r'$\log g$')
free_params_E['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_E['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')

parameters_E = Parameters42(free_params_E, constant_params_E)
parameters_E(np.random.rand(parameters_E.ndim))
parameters_E.params.update(best_fit_E)

# Per-chip median normalisation of the DATA — the model side is handled by
# passing normalize_flux='per_chip_median' to Retrieval42 below, which
# pRT_spectrum reads directly (self.normalize_flux) and applies inside
# make_spectrum() -> _apply_normalization(). No model_scale_factor needed
# (unlike §8's Ruffio case): the freshly generated model comes out already
# on the ~1.0 scale, matching the data below.
flux_E_N1, err_E_N1   = _pcm_1d(wave_N1, flux_N1, err_N1, _K2166_D)
flux_E_N2, err_E_N2   = _pcm_1d(wave_N2, flux_N2, err_N2, _K2166_D)
flux3_E_N1, err3_E_N1 = _pcm_3d(wave3_N1, flux3_N1, err3_N1, _K2166_D)
flux3_E_N2, err3_E_N2 = _pcm_3d(wave3_N2, flux3_N2, err3_N2, _K2166_D)

print(f'flux_E_N1 median: {np.nanmedian(flux_E_N1):.4f}  (should be ~1.0)')
print(f'flux_E_N2 median: {np.nanmedian(flux_E_N2):.4f}')

T1_E = Target42(wl=wave_N1, fl=flux_E_N1, err=err_E_N1, name='dh_tau_b_E_N1')
T2_E = Target42(wl=wave_N2, fl=flux_E_N2, err=err_E_N2, name='dh_tau_b_E_N2')

retrieval_E = Retrieval42(
    parameters         = parameters_E,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_E, T2_E],
    testing            = False,
    normalize_flux     = 'per_chip_median',
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = False,
    cov_mode           = 'GP',
)
retrieval_E.parameters.params.update(best_fit_E)
print('Retrieval 2968924 (008PM-EQ-MEDNORM, GP, v4.2) ready')

Best-fit params (2968924):
  rv_N1                     = 31.471
  rv_N2                     = 31.493
  vsini                     = 6.0073
  epsilon                   = 0.60422
  T_anchor                  = 1891
  dT_1                      = 534.42
  dT_2                      = 82.516
  dT_3                      = 78.265
  dT_4                      = 183.64
  dT_5                      = 176.8
  dT_6                      = 88.582
  dT_7                      = 328.08
  C_H                       = -0.25532
  C/O                       = 0.5968
  log_12CO_13CO             = 2.1599
  F_H                       = -0.92259
  log_Na                    = -4.0844
  log_Ca                    = -2.6497
  log_g                     = 3.6183
  log_a                     = 0.26703
  log_l                     = -2.2892
  [C/H]                     = -0.25532
  [C/H]_xsolar              = 0.5555
  s2                        = 1
  chi2                      = 1.0419
  chi2_N2                   = 0.95929
  lnZ  

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '14N-1H3' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/NH3/14N-1H3/14N-1H3__CoYuTe.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 2968924 (008PM-EQ-MEDNORM, GP, v4.2) ready


In [ ]:
results_E = run_species_ccf_validation(
    retrieval          = retrieval_E,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_E,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_E_N1,
    err_N1             = err3_E_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_E_N2,
    err_N2             = err3_E_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_E,
    retrieval_label    = LABEL_E,
    use_absolute_flux  = False,
)

## §10 — Retrieval 2027997  (007PM-EQ-GP, post-telluric-fix reference)

Cloud-free equilibrium chemistry + GP, absolute flux (log_M + log_R), N600, run on the
**corrected Night 1** flux-cal data (χ Tau telluric fix 2026-07-03). Post-fix analog of §7.

Unlike §5–§7 (which passed the raw sigmaclipper cubes), the observed cubes here are the
**flux-calibrated** products (W m⁻² μm⁻¹) — the same data the retrieval actually fit — so the
no-X residual subtraction operates in matching units, as `run_species_ccf_validation`'s
docstring specifies. SNR values are therefore not directly comparable to §5–§7's.

In [6]:
RID_F   = '2027997_N600_ev0.5_NormNone_PerChipScaleFalse'
DIR_F   = RETRIEVAL_BASE / RID_F
LABEL_F = '2027997'

# *42 aliases come from §7's import cell; fall back to _gb40 (same v4.2 file).
try:
    _ = make_free_params_equilibrium42
except NameError:
    Target42                       = _gb40.Target
    Parameters42                   = _gb40.Parameters
    Retrieval42                    = _gb40.Retrieval
    make_free_params_equilibrium42 = _gb40.make_free_params_equilibrium
    pRT_spectrum42                 = _gb40.pRT_spectrum
    print('  [fallback] *42 aliases set from _gb40 (same Guidebook v4.2 file)')
_GB = _gb42 if '_gb42' in globals() else _gb40

# 007-series data: ABSOLUTE flux-calibrated spectra (W m^-2 um^-1), corrected
# Night 1 telluric template (2026-07-03). 1-D for Target construction:
wave_F_N1, flux_F_N1, err_F_N1, R_F_N1 = _load_night40(
    '2022-12-31',
    flux_file        = 'extracted_spectra_combined_flux_cal.npy',
    err_file         = 'extracted_spectra_combined_err_flux_cal.npy',
    normalize_method = None,
)
wave_F_N2, flux_F_N2, err_F_N2, R_F_N2 = _load_night40(
    '2023-01-01',
    flux_file        = 'extracted_spectra_combined_flux_cal.npy',
    err_file         = 'extracted_spectra_combined_err_flux_cal.npy',
    normalize_method = None,
)


def _load_3d_fluxcal(night_str):
    """(3,5,2048) flux-cal cubes with NaN for bad pixels — for the CCF step."""
    base = Path(f'/data2/peng/{night_str}')
    flux = np.load(base / 'extracted_spectra_combined_flux_cal.npy').astype(float)
    err  = np.load(base / 'extracted_spectra_combined_err_flux_cal.npy').astype(float)
    hdu  = fits.open(base / 'cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')
    wave = np.array(hdu[1].data, dtype=float)[:, :5, :]
    bad  = ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~np.isfinite(wave)
    flux[bad] = np.nan; err[bad] = np.nan; wave[bad] = np.nan
    print(f'  [{night_str}] flux_cal 3-D: valid px {(~bad).sum()}  '
          f'median flux {np.nanmedian(flux):.3e} W/m2/um')
    return wave, flux, err


wave3_F_N1, flux3_F_N1, err3_F_N1 = _load_3d_fluxcal('2022-12-31')
wave3_F_N2, flux3_F_N2, err3_F_N2 = _load_3d_fluxcal('2023-01-01')

with open(DIR_F / 'final_params_dict.pickle', 'rb') as f:
    best_fit_F = pickle.load(f)
print('Best-fit params (2027997):')
for k, v in best_fit_F.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

# 007PM-EQ-GP config: equilibrium chem, absolute flux (log_M/log_R kept), GP —
# matches tasting_retrieval_equa_chem_v6.1_piette_cloudfree.py exactly.
constant_params_F = {'chemistry': 'equilibrium'}
free_params_F     = make_free_params_equilibrium42()
free_params_F['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_F['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')

parameters_F = Parameters42(free_params_F, constant_params_F)
parameters_F(np.random.rand(parameters_F.ndim))
parameters_F.params.update(best_fit_F)

T1_F = Target42(wl=wave_F_N1, fl=flux_F_N1, err=err_F_N1, name='dh_tau_b_F_N1')
T2_F = Target42(wl=wave_F_N2, fl=flux_F_N2, err=err_F_N2, name='dh_tau_b_F_N2')

retrieval_F = Retrieval42(
    parameters         = parameters_F,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_F, T2_F],
    testing            = False,
    normalize_flux     = None,
    per_chip_scaling   = False,
    instrument_res     = [R_F_N1, R_F_N2],
    use_absolute_flux  = True,
    cov_mode           = 'GP',
)
retrieval_F.parameters.params.update(best_fit_F)
print('Retrieval 2027997 (007PM-EQ-GP post-fix, v4.2) ready')

  [2022-12-31] No normalisation applied.
  [2022-12-31] Estimating resolving power...
  Estimated R = 314490 (median over 15 chips, range 297831–333720)
  [2022-12-31] Valid pixels: 26360 / 30720
  [2023-01-01] No normalisation applied.
  [2023-01-01] Estimating resolving power...
  Estimated R = 314480 (median over 15 chips, range 297860–333843)
  [2023-01-01] Valid pixels: 24964 / 30720
  [2022-12-31] flux_cal 3-D: valid px 26360  median flux 8.880e-16 W/m2/um
  [2023-01-01] flux_cal 3-D: valid px 24964  median flux 9.080e-16 W/m2/um
Best-fit params (2027997):
  rv_N1                     = 31.641
  rv_N2                     = 31.546
  vsini                     = 7.5393
  epsilon                   = 0.81952
  log_M                     = 1.2128
  log_R                     = 0.47929
  T_anchor                  = 2076.3
  dT_1                      = 592.29
  dT_2                      = 444.52
  dT_3                      = 60.827
  dT_4                      = 85.38
  dT_5                 

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


...

 Loading line opacities of species '12C-16O2__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO2/12C-16O2/12C-16O2__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 2027997 (007PM-EQ-GP post-fix, v4.2) ready


In [ ]:
results_F = run_species_ccf_validation(
    retrieval          = retrieval_F,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_F,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_F_N1,
    err_N1             = err3_F_N1,
    obs_wave_N1        = wave3_F_N1,
    obs_flux_N2        = flux3_F_N2,
    err_N2             = err3_F_N2,
    obs_wave_N2        = wave3_F_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_F,
    retrieval_label    = LABEL_F,
    use_absolute_flux  = True,
)

# Persist a machine-readable SNR summary (survives kernel kills; the printed
# output above is lost if nbconvert doesn't reach its final in-place write).
import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_F['ccf_results'].values()}
with open(DIR_F / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_F)

## §11 — Retrieval 2204420  (007PM-EQ-CLD, post-fix cloud pairing)

EddySed clouds (MgSiO₃ + Fe) + equilibrium chemistry + GP, absolute flux, N600, corrected
Night 1 data. Cloud analog of §10 (pairs with it like 1735916 paired with 2528367 pre-fix).

`cloud_species` is passed in `constant_params`, so the templates include cloud opacity —
**unlike legacy §5**, which validated cloud job 1735916 with cloud-free templates.
Reuses the flux-cal observation cubes loaded in §10.

Known caveat: this run's log_Ca posterior is bimodal (Ca–cloud degeneracy, see
`recording_recipe.md` 2026-07-03); the best-fit dict is used as-is.

In [7]:
RID_G   = '2204420_N600_ev0.5_NormNone_PerChipScaleFalse'
DIR_G   = RETRIEVAL_BASE / RID_G
LABEL_G = '2204420'

_CLOUD_SPECIES = list(_GB.CLOUD_SPECIES_DEFAULT)   # MgSiO3 + Fe (Xuan+2024)
print('cloud_species:', _CLOUD_SPECIES)

with open(DIR_G / 'final_params_dict.pickle', 'rb') as f:
    best_fit_G = pickle.load(f)
print('Best-fit params (2204420):')
for k, v in best_fit_G.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

# 007PM-EQ-CLD config: equilibrium + EddySed clouds + GP, absolute flux —
# matches tasting_retrieval_equa_chem_v6.1_piette_cloud.py exactly.
constant_params_G = {'chemistry': 'equilibrium', 'cloud_species': _CLOUD_SPECIES}
free_params_G     = _GB.make_free_params_equil_chem_cloudy(_CLOUD_SPECIES)
free_params_G['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_G['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')

parameters_G = Parameters42(free_params_G, constant_params_G)
parameters_G(np.random.rand(parameters_G.ndim))
parameters_G.params.update(best_fit_G)

T1_G = Target42(wl=wave_F_N1, fl=flux_F_N1, err=err_F_N1, name='dh_tau_b_G_N1')
T2_G = Target42(wl=wave_F_N2, fl=flux_F_N2, err=err_F_N2, name='dh_tau_b_G_N2')

retrieval_G = Retrieval42(
    parameters         = parameters_G,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_G, T2_G],
    testing            = False,
    normalize_flux     = None,
    per_chip_scaling   = False,
    instrument_res     = [R_F_N1, R_F_N2],
    use_absolute_flux  = True,
    cov_mode           = 'GP',
)
retrieval_G.parameters.params.update(best_fit_G)
print('Retrieval 2204420 (007PM-EQ-CLD post-fix, v4.2) ready')

cloud_species: ['MgSiO3(s)_crystalline_000', 'Fe(s)_crystalline_000']
Best-fit params (2204420):
  rv_N1                     = 31.633
  rv_N2                     = 31.593
  vsini                     = 7.7229
  epsilon                   = 0.8339
  log_M                     = 1.1937
  log_R                     = 0.4706
  T_anchor                  = 2127.8
  dT_1                      = 683.16
  dT_2                      = 430.36
  dT_3                      = 48.546
  dT_4                      = 157.62
  dT_5                      = 281.99
  dT_6                      = 115.81
  dT_7                      = 411.71
  C_H                       = -0.03974
  C/O                       = 0.64116
  log_12CO_13CO             = 1.6759
  F_H                       = -0.23265
  log_Na                    = -4.5137
  log_Ca                    = -5.3133
  log_X_MgSiO3              = -0.55636
  log_X_Fe                  = -1.0223
  fsed                      = 6.271
  log_Kzz                   = 11.752
  sigm

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


...

 Loading line opacities of species '12C-16O2__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO2/12C-16O2/12C-16O2__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
 Loading opacities of cloud species 'MgSiO3(s)_crystalline_000' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/clouds/MgSiO3(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000__

Retrieval 2204420 (007PM-EQ-CLD post-fix, v4.2) ready


In [ ]:
results_G = run_species_ccf_validation(
    retrieval          = retrieval_G,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_G,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_F_N1,
    err_N1             = err3_F_N1,
    obs_wave_N1        = wave3_F_N1,
    obs_flux_N2        = flux3_F_N2,
    err_N2             = err3_F_N2,
    obs_wave_N2        = wave3_F_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_G,
    retrieval_label    = LABEL_G,
    use_absolute_flux  = True,
)

# Persist a machine-readable SNR summary (survives kernel kills; the printed
# output above is lost if nbconvert doesn't reach its final in-place write).
import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_G['ccf_results'].values()}
with open(DIR_G / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_G)

## §12 — Retrieval 2499181  (007PM-EQ-CLD, N_live=1000 robustness re-run)

Identical config/data to §11 (2204420) but N_live 600→1000 — launched 2026-07-03 to check
that the log_Ca bimodality is a resolved likelihood feature, not an under-sampling artifact.
Same validation setup as §11; only the best-fit dict differs.

In [8]:
RID_H   = '2499181_N1000_ev0.5_NormNone_PerChipScaleFalse'
DIR_H   = RETRIEVAL_BASE / RID_H
LABEL_H = '2499181'

with open(DIR_H / 'final_params_dict.pickle', 'rb') as f:
    best_fit_H = pickle.load(f)
print('Best-fit params (2499181):')
for k, v in best_fit_H.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

constant_params_H = {'chemistry': 'equilibrium', 'cloud_species': _CLOUD_SPECIES}
free_params_H     = _GB.make_free_params_equil_chem_cloudy(_CLOUD_SPECIES)
free_params_H['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_H['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')

parameters_H = Parameters42(free_params_H, constant_params_H)
parameters_H(np.random.rand(parameters_H.ndim))
parameters_H.params.update(best_fit_H)

T1_H = Target42(wl=wave_F_N1, fl=flux_F_N1, err=err_F_N1, name='dh_tau_b_H_N1')
T2_H = Target42(wl=wave_F_N2, fl=flux_F_N2, err=err_F_N2, name='dh_tau_b_H_N2')

retrieval_H = Retrieval42(
    parameters         = parameters_H,
    N_live_points      = 1000,
    evidence_tolerance = 0.5,
    targets            = [T1_H, T2_H],
    testing            = False,
    normalize_flux     = None,
    per_chip_scaling   = False,
    instrument_res     = [R_F_N1, R_F_N2],
    use_absolute_flux  = True,
    cov_mode           = 'GP',
)
retrieval_H.parameters.params.update(best_fit_H)
print('Retrieval 2499181 (007PM-EQ-CLD N1000 post-fix, v4.2) ready')

Best-fit params (2499181):
  rv_N1                     = 31.551
  rv_N2                     = 31.568
  vsini                     = 7.5563
  epsilon                   = 0.75184
  log_M                     = 1.1811
  log_R                     = 0.47854
  T_anchor                  = 2097.6
  dT_1                      = 722.73
  dT_2                      = 437.19
  dT_3                      = 64.733
  dT_4                      = 121.83
  dT_5                      = 291.09
  dT_6                      = 134.64
  dT_7                      = 256.96
  C_H                       = 0.010131
  C/O                       = 0.65656
  log_12CO_13CO             = 1.7833
  F_H                       = -0.21586
  log_Na                    = -4.397
  log_Ca                    = -5.0394
  log_X_MgSiO3              = -0.30598
  log_X_Fe                  = -1.3003
  fsed                      = 5.6867
  log_Kzz                   = 11.047
  sigma_lnorm               = 1.9923
  log_a                     = 0.26633

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '14N-1H3' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/NH3/14N-1H3/14N-1H3__CoYuTe.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
 Loading opacities of cloud species 'MgSiO3(s)_crystalline_000' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/clouds/MgSiO3(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000__DHS.R39

Retrieval 2499181 (007PM-EQ-CLD N1000 post-fix, v4.2) ready


In [ ]:
results_H = run_species_ccf_validation(
    retrieval          = retrieval_H,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_H,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_F_N1,
    err_N1             = err3_F_N1,
    obs_wave_N1        = wave3_F_N1,
    obs_flux_N2        = flux3_F_N2,
    err_N2             = err3_F_N2,
    obs_wave_N2        = wave3_F_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_H,
    retrieval_label    = LABEL_H,
    use_absolute_flux  = True,
)

# Persist a machine-readable SNR summary (survives kernel kills; the printed
# output above is lost if nbconvert doesn't reach its final in-place write).
import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_H['ccf_results'].values()}
with open(DIR_H / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_H)

## §13 — Retrieval 3160171  (008PM-EQ-CLD-MEDNORM)

EddySed clouds + equilibrium chemistry + GP, **per-chip-median normalisation** (sigmaclipper
data, direct log_g, `use_absolute_flux=False`), N1000 — the cloud variant of §9 (2968924).
Data handling follows §9 exactly (fixed `_pcm_1d`/`_pcm_3d`, value-based per-chip medians;
no `model_scale_factor` needed — `normalize_flux='per_chip_median'` puts the model on the
data's ~1.0 scale inside `_apply_normalization`). Adds `cloud_species` so templates include
cloud opacity.

In [9]:
RID_I   = '3160171_N1000_ev0.5_Normper_chip_median_PerChipScaleFalse'
DIR_I   = RETRIEVAL_BASE / RID_I
LABEL_I = '3160171'

# Per-chip-median helpers — defined UNCONDITIONALLY (same rationale as §9: never
# trust a warm kernel to hold the fixed version).
_K2166_D = np.array([
    [[2063.711, 2077.942], [2078.967, 2092.559], [2093.479, 2106.392]],
    [[2143.087, 2157.855], [2158.914, 2173.020], [2173.983, 2187.386]],
    [[2228.786, 2244.133], [2245.229, 2259.888], [2260.904, 2274.835]],
    [[2321.596, 2337.568], [2338.704, 2353.961], [2355.035, 2369.534]],
    [[2422.415, 2439.061], [2440.243, 2456.145], [2457.275, 2472.388]],
])

def _pcm_1d(wave, flux, err, K2166):
    """Per-chip median normalisation for 1-D arrays (returns copies)."""
    flux = flux.copy(); err = err.copy()
    for order in range(5):
        for det in range(3):
            mask = (wave >= K2166[order, det, 0]) & (wave <= K2166[order, det, 1])
            if mask.sum() < 10: continue
            med = np.nanmedian(flux[mask])
            if med > 0: flux[mask] /= med; err[mask] /= med
    return flux, err

def _pcm_3d(wave3, flux3, err3, K2166):
    """Per-chip median normalisation for 3-D cubes; value-based (bugfix 2026-07-04:
    never map cube (det, order) indices to K2166 rows positionally — the order axis
    is reversed relative to the table)."""
    flux3 = flux3.copy(); err3 = err3.copy()
    for det in range(flux3.shape[0]):
        for order in range(flux3.shape[1]):
            fin = np.isfinite(flux3[det, order, :])
            if fin.sum() < 10:
                print(f'  [_pcm_3d] WARNING: det{det} ord{order}: only '
                      f'{fin.sum()} finite px — chip left un-normalised')
                continue
            med = np.nanmedian(flux3[det, order, fin])
            if med > 0: flux3[det, order, :] /= med; err3[det, order, :] /= med
    return flux3, err3

with open(DIR_I / 'final_params_dict.pickle', 'rb') as f:
    best_fit_I = pickle.load(f)
print('Best-fit params (3160171):')
for k, v in best_fit_I.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

# 008PM-EQ-CLD-MEDNORM config: equilibrium + EddySed clouds, direct log_g,
# per-chip-median norm, GP — matches
# tasting_retrieval_equa_chem_v6.1_piette_cloud_mednorm.py exactly.
constant_params_I = {'chemistry': 'equilibrium', 'cloud_species': _CLOUD_SPECIES}
free_params_I     = _GB.make_free_params_equil_chem_cloudy(_CLOUD_SPECIES)
del free_params_I['log_M']
del free_params_I['log_R']
free_params_I['log_g'] = ({'type': 'gaussian', 'mu': 3.64, 'sigma': 0.20}, r'$\log g$')
free_params_I['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_I['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')

parameters_I = Parameters42(free_params_I, constant_params_I)
parameters_I(np.random.rand(parameters_I.ndim))
parameters_I.params.update(best_fit_I)

# Sigmaclipper data, per-chip-median normalised (data side; model side handled
# by normalize_flux='per_chip_median' in the Retrieval below).
flux_I_N1, err_I_N1   = _pcm_1d(wave_N1, flux_N1, err_N1, _K2166_D)
flux_I_N2, err_I_N2   = _pcm_1d(wave_N2, flux_N2, err_N2, _K2166_D)
flux3_I_N1, err3_I_N1 = _pcm_3d(wave3_N1, flux3_N1, err3_N1, _K2166_D)
flux3_I_N2, err3_I_N2 = _pcm_3d(wave3_N2, flux3_N2, err3_N2, _K2166_D)

print(f'flux_I_N1 median: {np.nanmedian(flux_I_N1):.4f}  (should be ~1.0)')
print(f'flux_I_N2 median: {np.nanmedian(flux_I_N2):.4f}')

T1_I = Target42(wl=wave_N1, fl=flux_I_N1, err=err_I_N1, name='dh_tau_b_I_N1')
T2_I = Target42(wl=wave_N2, fl=flux_I_N2, err=err_I_N2, name='dh_tau_b_I_N2')

retrieval_I = Retrieval42(
    parameters         = parameters_I,
    N_live_points      = 1000,
    evidence_tolerance = 0.5,
    targets            = [T1_I, T2_I],
    testing            = False,
    normalize_flux     = 'per_chip_median',
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = False,
    cov_mode           = 'GP',
)
retrieval_I.parameters.params.update(best_fit_I)
print('Retrieval 3160171 (008PM-EQ-CLD-MEDNORM, v4.2) ready')

Best-fit params (3160171):
  rv_N1                     = 31.464
  rv_N2                     = 31.487
  vsini                     = 5.947
  epsilon                   = 0.52017
  T_anchor                  = 1911.7
  dT_1                      = 507.3
  dT_2                      = 91.188
  dT_3                      = 76.591
  dT_4                      = 175.15
  dT_5                      = 199.67
  dT_6                      = 91.886
  dT_7                      = 344.84
  C_H                       = -0.27755
  C/O                       = 0.59791
  log_12CO_13CO             = 2.1412
  F_H                       = -0.97147
  log_Na                    = -3.8251
  log_Ca                    = -3.0738
  log_X_MgSiO3              = -1.0559
  log_X_Fe                  = -1.4097
  fsed                      = 6.0033
  log_Kzz                   = 10.6
  sigma_lnorm               = 2.007
  log_g                     = 3.5726
  log_a                     = 0.26684
  log_l                     = -2.289
  [C/

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.


...

 Loading line opacities of species '12C-16O2__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO2/12C-16O2/12C-16O2__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
 Loading opacities of cloud species 'MgSiO3(s)_crystalline_000' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/clouds/MgSiO3(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000__

Retrieval 3160171 (008PM-EQ-CLD-MEDNORM, v4.2) ready


In [10]:
results_I = run_species_ccf_validation(
    retrieval          = retrieval_I,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_I,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_I_N1,
    err_N1             = err3_I_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_I_N2,
    err_N2             = err3_I_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_I,
    retrieval_label    = LABEL_I,
    use_absolute_flux  = False,
)

# Persist a machine-readable SNR summary (survives kernel kills; the printed
# output above is lost if nbconvert doesn't reach its final in-place write).
import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_I['ccf_results'].values()}
with open(DIR_I / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_I)

  Planet RV: N1 = +31.464 km/s  N2 = +31.487 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [4.85e-01, 1.33e+00]
  Generating no-X templates...


    [H2O] template RMS = 1.01e-01
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.87e-02


    [13CO] template RMS = 5.46e-03


...

  Saved: validation_deregt_HF.png


  Saved: validation_deregt_Na.png


  Saved: validation_deregt_Ca.png


  Saved: validation_deregt_totalCO.png


  Saved: validation_deregt_snr_summary.png
  CCF validation complete.
SNR summary JSON saved to /data2/peng/retrievals/3160171_N1000_ev0.5_Normper_chip_median_PerChipScaleFalse


---
## §14 — Retrieval 3497146  (008PM-EQ-MEDNORM, no Na/Ca)
**Guidebook v4.2 (2026-07-07 patch: Na/Ca commented out of EQ_SPECIES_PRT3) · GP covariance ·
Standard Gaussian likelihood (φ = 1 fixed) · N_live = 600**
Identical config to §9 (2968924) except Na and Ca are removed from the model entirely
(19 free params vs 21) — the A/B test for whether the noise-chasing atomics drive any of the
per-chip-median systematics. Data handling follows §9 exactly (fixed `_pcm_1d`/`_pcm_3d`,
value-based per-chip medians; no `model_scale_factor` needed —
`normalize_flux='per_chip_median'` puts the model on the data's ~1.0 scale inside
`_apply_normalization`). TRACE_SPECIES excludes Na/Ca (not in the model's mass fractions —
knockout templates would be no-ops).

| Parameter | MAP value |
|-----------|-----------|
| rv_N1     | 31.461 km/s |
| rv_N2     | 31.506 km/s |
| vsini     | 5.936 km/s |
| C/O       | 0.581 |
| log_g     | 3.678 |
| log_a     | 0.267 |
| log_l     | −2.289 |
| lnZ       | −18,064.3 (comparable to §9's −18,064.9 — same data scale, same series) |
| χ²        | 1.042 / 0.960 |


In [5]:
RID_J   = '3497146_N600_ev0.5_Normper_chip_median_PerChipScaleFalse'
DIR_J   = RETRIEVAL_BASE / RID_J
LABEL_J = '3497146'

# Reuse _gb42 / make_free_params_equilibrium42 / pRT_spectrum42 from §7.
# Fall back to _gb40 if §7 was skipped.
try:
    _ = make_free_params_equilibrium42
except NameError:
    Target42                       = _gb40.Target
    Parameters42                   = _gb40.Parameters
    Retrieval42                    = _gb40.Retrieval
    make_free_params_equilibrium42 = _gb40.make_free_params_equilibrium
    pRT_spectrum42                 = _gb40.pRT_spectrum
    print('  [fallback] *42 aliases set from _gb40 (same Guidebook v4.2 file)')

# No Na/Ca in this run's model (Guidebook 2026-07-07 patch) — knockout templates
# for them would be no-ops, so exclude them from the trace list explicitly.
TRACE_SPECIES_NONACA = {k: v for k, v in TRACE_SPECIES_BASE.items()
                        if v not in ('Na', 'Ca')}
print('TRACE_SPECIES_NONACA:', list(TRACE_SPECIES_NONACA.values()))

# Per-chip-median helpers — defined UNCONDITIONALLY (same rationale as §9: never
# trust a warm kernel to hold the fixed version).
_K2166_D = np.array([
    [[2063.711, 2077.942], [2078.967, 2092.559], [2093.479, 2106.392]],
    [[2143.087, 2157.855], [2158.914, 2173.020], [2173.983, 2187.386]],
    [[2228.786, 2244.133], [2245.229, 2259.888], [2260.904, 2274.835]],
    [[2321.596, 2337.568], [2338.704, 2353.961], [2355.035, 2369.534]],
    [[2422.415, 2439.061], [2440.243, 2456.145], [2457.275, 2472.388]],
])

def _pcm_1d(wave, flux, err, K2166):
    """Per-chip median normalisation for 1-D arrays (returns copies)."""
    flux = flux.copy(); err = err.copy()
    for order in range(5):
        for det in range(3):
            mask = (wave >= K2166[order, det, 0]) & (wave <= K2166[order, det, 1])
            if mask.sum() < 10: continue
            med = np.nanmedian(flux[mask])
            if med > 0: flux[mask] /= med; err[mask] /= med
    return flux, err

def _pcm_3d(wave3, flux3, err3, K2166):
    """Per-chip median normalisation for 3-D cubes; value-based (bugfix 2026-07-04:
    never map cube (det, order) indices to K2166 rows positionally — the order axis
    is reversed relative to the table)."""
    flux3 = flux3.copy(); err3 = err3.copy()
    for det in range(flux3.shape[0]):
        for order in range(flux3.shape[1]):
            fin = np.isfinite(flux3[det, order, :])
            if fin.sum() < 10:
                print(f'  [_pcm_3d] WARNING: det{det} ord{order}: only '
                      f'{fin.sum()} finite px — chip left un-normalised')
                continue
            med = np.nanmedian(flux3[det, order, fin])
            if med > 0: flux3[det, order, :] /= med; err3[det, order, :] /= med
    return flux3, err3

with open(DIR_J / 'final_params_dict.pickle', 'rb') as f:
    best_fit_J = pickle.load(f)
print('Best-fit params (3497146):')
for k, v in best_fit_J.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

# 008PM-EQ-MEDNORM (no Na/Ca) free params: equilibrium chem, direct log_g
# (no log_M/log_R), GP noise. The Guidebook's make_free_params_equilibrium no
# longer contains log_Na/log_Ca (2026-07-07 patch), so this construction gives
# the run's exact 19 params — matches
# tasting_retrieval_equa_chem_v6.1_piette_cloudfree_mednorm.py as launched 2026-07-07.
constant_params_J = {'chemistry': 'equilibrium'}
free_params_J     = make_free_params_equilibrium42()
del free_params_J['log_M']
del free_params_J['log_R']
free_params_J['log_g'] = ({'type': 'gaussian', 'mu': 3.64, 'sigma': 0.20}, r'$\log g$')
free_params_J['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_J['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')
assert 'log_Na' not in free_params_J and 'log_Ca' not in free_params_J, \
    'Guidebook on disk still has log_Na/log_Ca priors — wrong Guidebook state for 3497146'
print(f'free params: {len(free_params_J)} (expect 19)')

parameters_J = Parameters42(free_params_J, constant_params_J)
parameters_J(np.random.rand(parameters_J.ndim))
parameters_J.params.update(best_fit_J)

# Per-chip median normalisation of the DATA — the model side is handled by
# passing normalize_flux='per_chip_median' to Retrieval42 below (same as §9).
flux_J_N1, err_J_N1   = _pcm_1d(wave_N1, flux_N1, err_N1, _K2166_D)
flux_J_N2, err_J_N2   = _pcm_1d(wave_N2, flux_N2, err_N2, _K2166_D)
flux3_J_N1, err3_J_N1 = _pcm_3d(wave3_N1, flux3_N1, err3_N1, _K2166_D)
flux3_J_N2, err3_J_N2 = _pcm_3d(wave3_N2, flux3_N2, err3_N2, _K2166_D)

print(f'flux_J_N1 median: {np.nanmedian(flux_J_N1):.4f}  (should be ~1.0)')
print(f'flux_J_N2 median: {np.nanmedian(flux_J_N2):.4f}')

T1_J = Target42(wl=wave_N1, fl=flux_J_N1, err=err_J_N1, name='dh_tau_b_J_N1')
T2_J = Target42(wl=wave_N2, fl=flux_J_N2, err=err_J_N2, name='dh_tau_b_J_N2')

retrieval_J = Retrieval42(
    parameters         = parameters_J,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_J, T2_J],
    testing            = False,
    normalize_flux     = 'per_chip_median',
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = False,
    cov_mode           = 'GP',
)
retrieval_J.parameters.params.update(best_fit_J)
print('Retrieval 3497146 (008PM-EQ-MEDNORM no Na/Ca, GP, v4.2) ready')


  [fallback] *42 aliases set from _gb40 (same Guidebook v4.2 file)
TRACE_SPECIES_NONACA: ['H2O', '12CO', '13CO', 'CH4', 'FeH', 'HF']
Best-fit params (3497146):
  rv_N1                     = 31.461
  rv_N2                     = 31.506
  vsini                     = 5.9364
  epsilon                   = 0.54231
  T_anchor                  = 1856
  dT_1                      = 424
  dT_2                      = 84.176
  dT_3                      = 95.515
  dT_4                      = 188.72
  dT_5                      = 146.74
  dT_6                      = 91.199
  dT_7                      = 388.25
  C_H                       = -0.30609
  C/O                       = 0.58053
  log_12CO_13CO             = 2.1039
  F_H                       = -0.91979
  log_g                     = 3.6779
  log_a                     = 0.26704
  log_l                     = -2.289
  [C/H]                     = -0.30609
  [C/H]_xsolar              = 0.49421
  s2                        = 1
  chi2                    

Loading equilibrium chemistry table (done once)...
Loading chemical equilibrium chemistry table from file '/net/lem/data2/pRT3_formatted/input_data/pre_calculated_chemistry/equilibrium_chemistry/equilibrium_chemistry.chemtable.petitRADTRANS.h5'... 

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...


Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Done.
 Loading line opacities of species '12C-16O2__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO2/12C-16O2/12C-16O2__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 3497146 (008PM-EQ-MEDNORM no Na/Ca, GP, v4.2) ready


In [6]:
results_J = run_species_ccf_validation(
    retrieval          = retrieval_J,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_J,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_J_N1,
    err_N1             = err3_J_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_J_N2,
    err_N2             = err3_J_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_NONACA,
    retrieval_dir      = DIR_J,
    retrieval_label    = LABEL_J,
    use_absolute_flux  = False,
)

# Persist a machine-readable SNR summary (survives kernel kills; the printed
# output above is lost if nbconvert doesn't reach its final in-place write).
import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_J['ccf_results'].values()}
with open(DIR_J / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_J)


  Planet RV: N1 = +31.461 km/s  N2 = +31.506 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [4.90e-01, 1.33e+00]
  Generating no-X templates...


    [H2O] template RMS = 1.01e-01
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.85e-02


    [13CO] template RMS = 5.39e-03


...

  Saved: validation_deregt_CH4.png


  Saved: validation_deregt_FeH.png


  Saved: validation_deregt_HF.png


  Saved: validation_deregt_totalCO.png


  Saved: validation_deregt_snr_summary.png
  CCF validation complete.
SNR summary JSON saved to /data2/peng/retrievals/3497146_N600_ev0.5_Normper_chip_median_PerChipScaleFalse


---
## §15 — Retrieval 3708914  (008PM-EQ-CLD-MEDNORM, no Na/Ca)
**Guidebook v4.2 (2026-07-07 patch: Na/Ca commented out of EQ_SPECIES_PRT3) · GP covariance ·
Standard Gaussian likelihood (φ = 1 fixed) · EddySed MgSiO₃+Fe clouds · N_live = 1000**
Identical config to §13 (3160171) except Na and Ca are removed from the model entirely
(24 free params vs 26) — the cloud branch of the §14 A/B test. Completes the 2×2 mednorm
matrix (cloud-free/cloud × with/without Na+Ca): 2968924 / 3160171 / 3497146 / 3708914.
Data handling follows §13/§9 exactly (fixed `_pcm_1d`/`_pcm_3d`, value-based per-chip
medians; no `model_scale_factor` needed — `normalize_flux='per_chip_median'` puts the model
on the data's ~1.0 scale inside `_apply_normalization`). TRACE_SPECIES excludes Na/Ca (not
in the model's mass fractions — knockout templates would be no-ops).

| Parameter | MAP value |
|-----------|-----------|
| rv_N1     | 31.462 km/s |
| rv_N2     | 31.517 km/s |
| vsini     | 5.968 km/s |
| C/O       | 0.577 |
| log_g     | 3.685 |
| log_X_MgSiO3 | −0.835 |
| log_X_Fe  | −1.411 |
| fsed      | 4.75 |
| log_Kzz   | 10.40 |
| sigma_lnorm | 2.12 |
| log_a     | 0.267 |
| log_l     | −2.289 |
| lnZ       | −18,072.8 (INS; §13's 3160171: −18,074.6 — same data scale, same series) |
| χ²        | 1.042 / 0.959 |


In [5]:
RID_K   = '3708914_N1000_ev0.5_Normper_chip_median_PerChipScaleFalse'
DIR_K   = RETRIEVAL_BASE / RID_K
LABEL_K = '3708914'

# Reuse _gb42 / pRT_spectrum42 aliases from §7; fall back to _gb40 (the same
# Guidebook v4.2 file) if §7 was skipped this session.
try:
    _ = pRT_spectrum42
except NameError:
    Target42                       = _gb40.Target
    Parameters42                   = _gb40.Parameters
    Retrieval42                    = _gb40.Retrieval
    make_free_params_equilibrium42 = _gb40.make_free_params_equilibrium
    pRT_spectrum42                 = _gb40.pRT_spectrum
    print('  [fallback] *42 aliases set from _gb40 (same Guidebook v4.2 file)')

_GB_K = _gb42 if '_gb42' in globals() else _gb40
_CLOUD_SPECIES_K = list(_GB_K.CLOUD_SPECIES_DEFAULT)   # MgSiO3 + Fe (Xuan+2024)
print('cloud_species:', _CLOUD_SPECIES_K)

# No Na/Ca in this run's model (Guidebook 2026-07-07 patch) — knockout templates
# for them would be no-ops, so exclude them from the trace list explicitly.
TRACE_SPECIES_NONACA = {k: v for k, v in TRACE_SPECIES_BASE.items()
                        if v not in ('Na', 'Ca')}
print('TRACE_SPECIES_NONACA:', list(TRACE_SPECIES_NONACA.values()))

# Per-chip-median helpers — defined UNCONDITIONALLY (same rationale as §9: never
# trust a warm kernel to hold the fixed version).
_K2166_D = np.array([
    [[2063.711, 2077.942], [2078.967, 2092.559], [2093.479, 2106.392]],
    [[2143.087, 2157.855], [2158.914, 2173.020], [2173.983, 2187.386]],
    [[2228.786, 2244.133], [2245.229, 2259.888], [2260.904, 2274.835]],
    [[2321.596, 2337.568], [2338.704, 2353.961], [2355.035, 2369.534]],
    [[2422.415, 2439.061], [2440.243, 2456.145], [2457.275, 2472.388]],
])

def _pcm_1d(wave, flux, err, K2166):
    """Per-chip median normalisation for 1-D arrays (returns copies)."""
    flux = flux.copy(); err = err.copy()
    for order in range(5):
        for det in range(3):
            mask = (wave >= K2166[order, det, 0]) & (wave <= K2166[order, det, 1])
            if mask.sum() < 10: continue
            med = np.nanmedian(flux[mask])
            if med > 0: flux[mask] /= med; err[mask] /= med
    return flux, err

def _pcm_3d(wave3, flux3, err3, K2166):
    """Per-chip median normalisation for 3-D cubes; value-based (bugfix 2026-07-04:
    never map cube (det, order) indices to K2166 rows positionally — the order axis
    is reversed relative to the table)."""
    flux3 = flux3.copy(); err3 = err3.copy()
    for det in range(flux3.shape[0]):
        for order in range(flux3.shape[1]):
            fin = np.isfinite(flux3[det, order, :])
            if fin.sum() < 10:
                print(f'  [_pcm_3d] WARNING: det{det} ord{order}: only '
                      f'{fin.sum()} finite px — chip left un-normalised')
                continue
            med = np.nanmedian(flux3[det, order, fin])
            if med > 0: flux3[det, order, :] /= med; err3[det, order, :] /= med
    return flux3, err3

with open(DIR_K / 'final_params_dict.pickle', 'rb') as f:
    best_fit_K = pickle.load(f)
print('Best-fit params (3708914):')
for k, v in best_fit_K.items():
    if np.ndim(v) == 0 and v is not None:
        print(f'  {k:25s} = {float(v):.5g}')

# 008PM-EQ-CLD-MEDNORM (no Na/Ca) config: equilibrium + EddySed clouds, direct
# log_g, per-chip-median norm, GP — matches
# tasting_retrieval_equa_chem_v6.1_piette_cloud_mednorm.py as launched 2026-07-08.
# make_free_params_equil_chem_cloudy builds on make_free_params_equilibrium,
# which no longer contains log_Na/log_Ca (2026-07-07 patch) -> 24 params.
constant_params_K = {'chemistry': 'equilibrium', 'cloud_species': _CLOUD_SPECIES_K}
free_params_K     = _GB_K.make_free_params_equil_chem_cloudy(_CLOUD_SPECIES_K)
del free_params_K['log_M']
del free_params_K['log_R']
free_params_K['log_g'] = ({'type': 'gaussian', 'mu': 3.64, 'sigma': 0.20}, r'$\log g$')
free_params_K['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_K['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')
assert 'log_Na' not in free_params_K and 'log_Ca' not in free_params_K, \
    'Guidebook on disk still has log_Na/log_Ca priors — wrong Guidebook state for 3708914'
print(f'free params: {len(free_params_K)} (expect 24)')

parameters_K = Parameters42(free_params_K, constant_params_K)
parameters_K(np.random.rand(parameters_K.ndim))
parameters_K.params.update(best_fit_K)

# Sigmaclipper data, per-chip-median normalised (data side; model side handled
# by normalize_flux='per_chip_median' in the Retrieval below).
flux_K_N1, err_K_N1   = _pcm_1d(wave_N1, flux_N1, err_N1, _K2166_D)
flux_K_N2, err_K_N2   = _pcm_1d(wave_N2, flux_N2, err_N2, _K2166_D)
flux3_K_N1, err3_K_N1 = _pcm_3d(wave3_N1, flux3_N1, err3_N1, _K2166_D)
flux3_K_N2, err3_K_N2 = _pcm_3d(wave3_N2, flux3_N2, err3_N2, _K2166_D)

print(f'flux_K_N1 median: {np.nanmedian(flux_K_N1):.4f}  (should be ~1.0)')
print(f'flux_K_N2 median: {np.nanmedian(flux_K_N2):.4f}')

T1_K = Target42(wl=wave_N1, fl=flux_K_N1, err=err_K_N1, name='dh_tau_b_K_N1')
T2_K = Target42(wl=wave_N2, fl=flux_K_N2, err=err_K_N2, name='dh_tau_b_K_N2')

retrieval_K = Retrieval42(
    parameters         = parameters_K,
    N_live_points      = 1000,
    evidence_tolerance = 0.5,
    targets            = [T1_K, T2_K],
    testing            = False,
    normalize_flux     = 'per_chip_median',
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = False,
    cov_mode           = 'GP',
)
retrieval_K.parameters.params.update(best_fit_K)
print('Retrieval 3708914 (008PM-EQ-CLD-MEDNORM no Na/Ca, GP, v4.2) ready')

  [fallback] *42 aliases set from _gb40 (same Guidebook v4.2 file)
cloud_species: ['MgSiO3(s)_crystalline_000', 'Fe(s)_crystalline_000']
TRACE_SPECIES_NONACA: ['H2O', '12CO', '13CO', 'CH4', 'FeH', 'HF']
Best-fit params (3708914):
  rv_N1                     = 31.462
  rv_N2                     = 31.517
  vsini                     = 5.9676
  epsilon                   = 0.57055
  T_anchor                  = 1885.8
  dT_1                      = 461.57
  dT_2                      = 113.55
  dT_3                      = 90.942
  dT_4                      = 220.54
  dT_5                      = 134.99
  dT_6                      = 96.233
  dT_7                      = 529.89
  C_H                       = -0.32039
  C/O                       = 0.57725
  log_12CO_13CO             = 2.1168
  F_H                       = -0.89255
  log_X_MgSiO3              = -0.83499
  log_X_Fe                  = -1.4111
  fsed                      = 4.7549
  log_Kzz                   = 10.399
  sigma_lnorm        

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Done.
 Loading line opacities of species '12C-16O2__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO2/12C-16O2/12C-16O2__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
 Loading opacities of cloud species 'MgSiO3(s)_crystalline_000' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/clouds/MgSiO3(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000__DHS.R39_0.1-250mu.cotable.petitRADTRANS.h5' (crystalline_000, using DHS scattering)... Done.
 Loading opacities of cloud species 'Fe(s)_crystalline_000' from file '/net/lem/data2/pRT3_formatted/

Retrieval 3708914 (008PM-EQ-CLD-MEDNORM no Na/Ca, GP, v4.2) ready


In [6]:
results_K = run_species_ccf_validation(
    retrieval          = retrieval_K,
    pRT_spectrum_class = pRT_spectrum42,
    best_fit_params    = best_fit_K,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_K_N1,
    err_N1             = err3_K_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_K_N2,
    err_N2             = err3_K_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_NONACA,
    retrieval_dir      = DIR_K,
    retrieval_label    = LABEL_K,
    use_absolute_flux  = False,
)

# Persist a machine-readable SNR summary (survives kernel kills; the printed
# output above is lost if nbconvert doesn't reach its final in-place write).
import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_K['ccf_results'].values()}
with open(DIR_K / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_K)

  Planet RV: N1 = +31.462 km/s  N2 = +31.517 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [4.85e-01, 1.34e+00]
  Generating no-X templates...


    [H2O] template RMS = 1.03e-01
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.92e-02


    [13CO] template RMS = 5.25e-03


...

  Saved: validation_deregt_13CO.png


  Saved: validation_deregt_CH4.png


  Saved: validation_deregt_FeH.png


  Saved: validation_deregt_HF.png


  Saved: validation_deregt_totalCO.png
  Saved: validation_deregt_snr_summary.png
  CCF validation complete.
SNR summary JSON saved to /data2/peng/retrievals/3708914_N1000_ev0.5_Normper_chip_median_PerChipScaleFalse


---

## §16 — Retrieval 51459 (008PM-EQ-MEDNORM-ATOMOPAC): first real CCF test of Na+Ca opacity

First job in this project where Na and Ca are genuine opacity sources in the model spectrum (Guidebook v4.2.5, zero new free parameters — see recording_recipe.md 2026-07-20/21). RID_J (3497146) and RID_K (3708914) both excluded Na/Ca from TRACE_SPECIES because those runs' models never contained Na/Ca opacity at all (`_resolve_mf_key` would have returned None and the no-X template would have been a no-op). Here we use the FULL `TRACE_SPECIES_BASE` (includes Na, Ca) against a Guidebook v4.2.5-built Retrieval object, so the no-X templates for Na/Ca are real.

In [5]:
# Import Guidebook v4.2.5 (Na+Ca opacity restored, zero new free params) — needed so
# EQ_SPECIES_PRT3 actually contains '23Na'/'40Ca' and equilibrium_chemistry() computes
# their mass fractions. v4.2 (_gb40/_gb42 above) has both commented out — reusing it
# here would silently make the Na/Ca CCF a no-op again, as it was for RID_J/RID_K.
_gb425_path = str(RECIPE_DIR / 'Tasting_guidebook' / 'Guidebook_GAStronomy_Piette_v4.2.5.py')
_spec425 = importlib.util.spec_from_file_location('Guidebook_v4_2_5', _gb425_path)
_gb425   = importlib.util.module_from_spec(_spec425)
_spec425.loader.exec_module(_gb425)

Target425                       = _gb425.Target
Parameters425                   = _gb425.Parameters
Retrieval425                    = _gb425.Retrieval
make_free_params_equilibrium425 = _gb425.make_free_params_equilibrium
pRT_spectrum425                 = _gb425.pRT_spectrum

print('Guidebook v4.2.5 loaded; EQ_SPECIES_PRT3 =', _gb425.EQ_SPECIES_PRT3)
assert '23Na' in _gb425.EQ_SPECIES_PRT3 and '40Ca' in _gb425.EQ_SPECIES_PRT3, \
    'v4.2.5 must carry Na and Ca in EQ_SPECIES_PRT3'


Input data path changed to '/net/lem/data2/pRT3_formatted/input_data'
Guidebook v4.2.5 loaded; EQ_SPECIES_PRT3 = ['1H2-16O', '12C-16O__HITEMP', '13C-16O', '12C-1H4__MM', '14N-1H3', '1H2-32S', '1H-12C-14N', '12C-16O2__HITEMP', '56Fe-1H', '1H-19F', '23Na', '40Ca']


In [6]:
RID_L   = '51459_N600_ev0.5_Normper_chip_median_PerChipScaleFalse'
DIR_L   = RETRIEVAL_BASE / RID_L
LABEL_L = '51459'

# Na and Ca ARE in this run's model (Guidebook v4.2.5) — unlike RID_J/RID_K, use the
# FULL trace list so the CCF genuinely tests for them.
print('TRACE_SPECIES_BASE:', list(TRACE_SPECIES_BASE.values()))

with open(DIR_L / 'final_params_dict.pickle', 'rb') as f:
    best_fit_L = pickle.load(f)
print('Best-fit params (51459):')
for k, v in best_fit_L.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

constant_params_L = {'chemistry': 'equilibrium'}
free_params_L     = make_free_params_equilibrium425()
del free_params_L['log_M']
del free_params_L['log_R']
free_params_L['log_g'] = ({'type': 'gaussian', 'mu': 3.64, 'sigma': 0.20}, r'$\log g$')
free_params_L['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_L['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')
assert 'log_Na' not in free_params_L and 'log_Ca' not in free_params_L, \
    'Guidebook v4.2.5 must NOT carry log_Na/log_Ca as free params'
print(f'free params: {len(free_params_L)} (expect 19)')

parameters_L = Parameters425(free_params_L, constant_params_L)
parameters_L(np.random.rand(parameters_L.ndim))
parameters_L.params.update(best_fit_L)

# Local per-chip-median helpers (self-contained — not relying on RID_J/K's cell
# having been executed in this run).
_K2166_L = np.array([
    [[2063.711, 2077.942], [2078.967, 2092.559], [2093.479, 2106.392]],
    [[2143.087, 2157.855], [2158.914, 2173.020], [2173.983, 2187.386]],
    [[2228.786, 2244.133], [2245.229, 2259.888], [2260.904, 2274.835]],
    [[2321.596, 2337.568], [2338.704, 2353.961], [2355.035, 2369.534]],
    [[2422.415, 2439.061], [2440.243, 2456.145], [2457.275, 2472.388]],
])

def _pcm_1d_L(wave, flux, err, K2166):
    flux = flux.copy(); err = err.copy()
    for order in range(5):
        for det in range(3):
            mask = (wave >= K2166[order, det, 0]) & (wave <= K2166[order, det, 1])
            if mask.sum() < 10: continue
            med = np.nanmedian(flux[mask])
            if med > 0: flux[mask] /= med; err[mask] /= med
    return flux, err

def _pcm_3d_L(wave3, flux3, err3, K2166):
    flux3 = flux3.copy(); err3 = err3.copy()
    for det in range(flux3.shape[0]):
        for order in range(flux3.shape[1]):
            fin = np.isfinite(flux3[det, order, :])
            if fin.sum() < 10:
                print(f'  [_pcm_3d_L] WARNING: det{det} ord{order}: only '
                      f'{fin.sum()} finite px — chip left un-normalised')
                continue
            med = np.nanmedian(flux3[det, order, fin])
            if med > 0: flux3[det, order, :] /= med; err3[det, order, :] /= med
    return flux3, err3

flux_L_N1, err_L_N1   = _pcm_1d_L(wave_N1, flux_N1, err_N1, _K2166_L)
flux_L_N2, err_L_N2   = _pcm_1d_L(wave_N2, flux_N2, err_N2, _K2166_L)
flux3_L_N1, err3_L_N1 = _pcm_3d_L(wave3_N1, flux3_N1, err3_N1, _K2166_L)
flux3_L_N2, err3_L_N2 = _pcm_3d_L(wave3_N2, flux3_N2, err3_N2, _K2166_L)

print(f'flux_L_N1 median: {np.nanmedian(flux_L_N1):.4f}  (should be ~1.0)')
print(f'flux_L_N2 median: {np.nanmedian(flux_L_N2):.4f}')

T1_L = Target425(wl=wave_N1, fl=flux_L_N1, err=err_L_N1, name='dh_tau_b_L_N1')
T2_L = Target425(wl=wave_N2, fl=flux_L_N2, err=err_L_N2, name='dh_tau_b_L_N2')

retrieval_L = Retrieval425(
    parameters         = parameters_L,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_L, T2_L],
    testing            = False,
    normalize_flux     = 'per_chip_median',
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = False,
    cov_mode           = 'GP',
)
retrieval_L.parameters.params.update(best_fit_L)
print('Retrieval 51459 (008PM-EQ-MEDNORM-ATOMOPAC, Na+Ca opacity, GP, v4.2.5) ready')


TRACE_SPECIES_BASE: ['H2O', '12CO', '13CO', 'CH4', 'FeH', 'HF', 'Na', 'Ca']
Best-fit params (51459):
  rv_N1                     = 31.47
  rv_N2                     = 31.508
  vsini                     = 5.9106
  epsilon                   = 0.50044
  T_anchor                  = 1891.6
  dT_1                      = 423.3
  dT_2                      = 86.031
  dT_3                      = 72.153
  dT_4                      = 196.17
  dT_5                      = 164.86
  dT_6                      = 83.532
  dT_7                      = 346.78
  C_H                       = -0.27049
  C/O                       = 0.5869
  log_12CO_13CO             = 2.1507
  F_H                       = -0.89448
  log_g                     = 3.6788
  log_a                     = 0.267
  log_l                     = -2.2894
  [C/H]                     = -0.27049
  [C/H]_xsolar              = 0.53642
  s2                        = 1
  chi2                      = 1.0418
  chi2_N2                   = 0.95967
  lnZ    

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Done.
 Loading line opacities of species '56Fe-1H' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/FeH/56Fe-1H/56Fe-1H__MoLLIST.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'... Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 51459 (008PM-EQ-MEDNORM-ATOMOPAC, Na+Ca opacity, GP, v4.2.5) ready


In [7]:
results_L = run_species_ccf_validation(
    retrieval          = retrieval_L,
    pRT_spectrum_class = pRT_spectrum425,
    best_fit_params    = best_fit_L,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_L_N1,
    err_N1             = err3_L_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_L_N2,
    err_N2             = err3_L_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_L,
    retrieval_label    = LABEL_L,
    use_absolute_flux  = False,
)

import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_L['ccf_results'].values()}
with open(DIR_L / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_L)

print()
print('=== Na / Ca CCF significance (first real test — opacity now genuinely in the model) ===')
for sp in ('Na', 'Ca'):
    found = False
    for prt_name, res in results_L['ccf_results'].items():
        if res['label'] == sp:
            found = True
            print(f"  {sp:3s}: peak CCF = {res['peak_val']:.3e}   "
                  f"peak_rv = {res['peak_rv']:+.1f} km/s   SNR = {res['snr']:.2f}   "
                  f"template_fraction = {res['template_fraction']:.1e}")
    if not found:
        print(f'  {sp:3s}: NOT in ccf_results (template negligible / not resolved in mass_fractions)')


  Planet RV: N1 = +31.470 km/s  N2 = +31.508 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [4.94e-01, 1.32e+00]
  Generating no-X templates...


    [H2O] template RMS = 9.96e-02
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.78e-02


    [13CO] template RMS = 4.95e-03


...

  Saved: validation_deregt_HF.png


  Saved: validation_deregt_Na.png


  Saved: validation_deregt_Ca.png


  Saved: validation_deregt_totalCO.png


  Saved: validation_deregt_snr_summary.png
  CCF validation complete.
SNR summary JSON saved to /data2/peng/retrievals/51459_N600_ev0.5_Normper_chip_median_PerChipScaleFalse

=== Na / Ca CCF significance (first real test — opacity now genuinely in the model) ===
  Na : peak CCF = 7.786e-01   peak_rv = -337.0 km/s   SNR = 4.25   template_fraction = 5.4e-03
  Ca : peak CCF = 2.136e-01   peak_rv = +540.0 km/s   SNR = 3.00   template_fraction = 2.3e-03


---

## §17 — Retrieval 430784 (008PM-EQ-MEDNORM-ATOMOPAC, successor to 51459): Na+Ca CCF check

Same script/Guidebook v4.2.5 config as RID_L/51459 (19 free params, Na+Ca opacity, zero new free params). User intended this as a dT-prior-widened successor to 51459 — verified on disk that `make_free_params_equilibrium()`'s dT_i bounds are unchanged from 51459's at time of writing (see `recording_recipe.md` 2026-07-21/22), so treat this as a robustness repeat of 51459 rather than a confirmed distinct prior-sensitivity test. Same TRACE_SPECIES_BASE (includes Na, Ca) as §16.

In [6]:
RID_M   = '430784_N600_ev0.5_Normper_chip_median_PerChipScaleFalse'
DIR_M   = RETRIEVAL_BASE / RID_M
LABEL_M = '430784'

print('TRACE_SPECIES_BASE:', list(TRACE_SPECIES_BASE.values()))

with open(DIR_M / 'final_params_dict.pickle', 'rb') as f:
    best_fit_M = pickle.load(f)
print('Best-fit params (430784):')
for k, v in best_fit_M.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')

constant_params_M = {'chemistry': 'equilibrium'}
free_params_M     = make_free_params_equilibrium425()
del free_params_M['log_M']
del free_params_M['log_R']
free_params_M['log_g'] = ({'type': 'gaussian', 'mu': 3.64, 'sigma': 0.20}, r'$\log g$')
free_params_M['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_M['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')
assert 'log_Na' not in free_params_M and 'log_Ca' not in free_params_M, \
    'Guidebook v4.2.5 must NOT carry log_Na/log_Ca as free params'
print(f'free params: {len(free_params_M)} (expect 19)')

parameters_M = Parameters425(free_params_M, constant_params_M)
parameters_M(np.random.rand(parameters_M.ndim))
parameters_M.params.update(best_fit_M)

_K2166_M = np.array([
    [[2063.711, 2077.942], [2078.967, 2092.559], [2093.479, 2106.392]],
    [[2143.087, 2157.855], [2158.914, 2173.020], [2173.983, 2187.386]],
    [[2228.786, 2244.133], [2245.229, 2259.888], [2260.904, 2274.835]],
    [[2321.596, 2337.568], [2338.704, 2353.961], [2355.035, 2369.534]],
    [[2422.415, 2439.061], [2440.243, 2456.145], [2457.275, 2472.388]],
])

def _pcm_1d_M(wave, flux, err, K2166):
    flux = flux.copy(); err = err.copy()
    for order in range(5):
        for det in range(3):
            mask = (wave >= K2166[order, det, 0]) & (wave <= K2166[order, det, 1])
            if mask.sum() < 10: continue
            med = np.nanmedian(flux[mask])
            if med > 0: flux[mask] /= med; err[mask] /= med
    return flux, err

def _pcm_3d_M(wave3, flux3, err3, K2166):
    flux3 = flux3.copy(); err3 = err3.copy()
    for det in range(flux3.shape[0]):
        for order in range(flux3.shape[1]):
            fin = np.isfinite(flux3[det, order, :])
            if fin.sum() < 10:
                print(f'  [_pcm_3d_M] WARNING: det{det} ord{order}: only '
                      f'{fin.sum()} finite px — chip left un-normalised')
                continue
            med = np.nanmedian(flux3[det, order, fin])
            if med > 0: flux3[det, order, :] /= med; err3[det, order, :] /= med
    return flux3, err3

flux_M_N1, err_M_N1   = _pcm_1d_M(wave_N1, flux_N1, err_N1, _K2166_M)
flux_M_N2, err_M_N2   = _pcm_1d_M(wave_N2, flux_N2, err_N2, _K2166_M)
flux3_M_N1, err3_M_N1 = _pcm_3d_M(wave3_N1, flux3_N1, err3_N1, _K2166_M)
flux3_M_N2, err3_M_N2 = _pcm_3d_M(wave3_N2, flux3_N2, err3_N2, _K2166_M)

print(f'flux_M_N1 median: {np.nanmedian(flux_M_N1):.4f}  (should be ~1.0)')
print(f'flux_M_N2 median: {np.nanmedian(flux_M_N2):.4f}')

T1_M = Target425(wl=wave_N1, fl=flux_M_N1, err=err_M_N1, name='dh_tau_b_M_N1')
T2_M = Target425(wl=wave_N2, fl=flux_M_N2, err=err_M_N2, name='dh_tau_b_M_N2')

retrieval_M = Retrieval425(
    parameters         = parameters_M,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_M, T2_M],
    testing            = False,
    normalize_flux     = 'per_chip_median',
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = False,
    cov_mode           = 'GP',
)
retrieval_M.parameters.params.update(best_fit_M)
print('Retrieval 430784 (008PM-EQ-MEDNORM-ATOMOPAC, Na+Ca opacity, GP, v4.2.5) ready')


TRACE_SPECIES_BASE: ['H2O', '12CO', '13CO', 'CH4', 'FeH', 'HF', 'Na', 'Ca']
Best-fit params (430784):
  rv_N1                     = 31.453
  rv_N2                     = 31.494
  vsini                     = 5.9915
  epsilon                   = 0.59453
  T_anchor                  = 1871
  dT_1                      = 415.97
  dT_2                      = 73.201
  dT_3                      = 95.638
  dT_4                      = 179.75
  dT_5                      = 161.62
  dT_6                      = 86.018
  dT_7                      = 326.46
  C_H                       = -0.25529
  C/O                       = 0.58688
  log_12CO_13CO             = 2.1507
  F_H                       = -0.93528
  log_g                     = 3.6677
  log_a                     = 0.26699
  log_l                     = -2.2894
  [C/H]                     = -0.25529
  [C/H]_xsolar              = 0.55554
  s2                        = 1
  chi2                      = 1.0419
  chi2_N2                   = 0.95956
  lnZ

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '13C-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/13C-16O/13C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-1H4__MM' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CH4/12C-1H4/12C-1H4__MM.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
Successfully loaded all opacities


Retrieval 430784 (008PM-EQ-MEDNORM-ATOMOPAC, Na+Ca opacity, GP, v4.2.5) ready


In [7]:
results_M = run_species_ccf_validation(
    retrieval          = retrieval_M,
    pRT_spectrum_class = pRT_spectrum425,
    best_fit_params    = best_fit_M,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_M_N1,
    err_N1             = err3_M_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_M_N2,
    err_N2             = err3_M_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_M,
    retrieval_label    = LABEL_M,
    use_absolute_flux  = False,
)

import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_M['ccf_results'].values()}
with open(DIR_M / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_M)

print()
print('=== Na / Ca CCF significance — 430784 ===')
for sp in ('Na', 'Ca'):
    found = False
    for prt_name, res in results_M['ccf_results'].items():
        if res['label'] == sp:
            found = True
            print(f"  {sp:3s}: peak CCF = {res['peak_val']:.3e}   "
                  f"peak_rv = {res['peak_rv']:+.1f} km/s   SNR = {res['snr']:.2f}   "
                  f"template_fraction = {res['template_fraction']:.1e}")
    if not found:
        print(f'  {sp:3s}: NOT in ccf_results (template negligible / not resolved in mass_fractions)')


  Planet RV: N1 = +31.453 km/s  N2 = +31.494 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [4.99e-01, 1.33e+00]
  Generating no-X templates...


    [H2O] template RMS = 1.02e-01
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.87e-02


    [13CO] template RMS = 5.18e-03


...

  Saved: validation_deregt_HF.png


  Saved: validation_deregt_Na.png


  Saved: validation_deregt_Ca.png


  Saved: validation_deregt_totalCO.png


  Saved: validation_deregt_snr_summary.png
  CCF validation complete.
SNR summary JSON saved to /data2/peng/retrievals/430784_N600_ev0.5_Normper_chip_median_PerChipScaleFalse

=== Na / Ca CCF significance — 430784 ===
  Na : peak CCF = 7.317e-01   peak_rv = -337.0 km/s   SNR = 4.22   template_fraction = 5.0e-03
  Ca : peak CCF = 1.716e-01   peak_rv = +999.0 km/s   SNR = 3.04   template_fraction = 1.8e-03


---
## §18 — Retrieval 1001064 (008PM-EQ-CLD-MEDNORM-ATOMOPAC): cloud counterpart of RID_L/51459

Cloud counterpart of §16's 51459 -- same relationship as §15's 3708914 was to §9's 2968924 / §14's 3497146 (cloud vs cloud-free, no-Na/Ca counterpart). Guidebook v4.2.5 (Na/Ca opacity restored, zero new free params), EddySed MgSiO3+Fe clouds, GP covariance, per-chip-median normalisation, 24 free params. Uses the FULL `TRACE_SPECIES_BASE` (includes Na, Ca) since this run's model genuinely contains their opacity, same rationale as §16/17.

Job completed 2026-07-22 but was never analysed until now (its own `retrieval_model_*.npy` post-processing was missing on disk -- OOM-killed like 3708914 was, see `recording_recipe.md` §2026-08-05 -- regenerated via a single-process `evaluate()` call before this section was written). Promoted to the standing cloud benchmark 2026-08-05 per the user's explicit decision that benchmarks must include Na/Ca opacity without sampling them as free parameters.

In [6]:
RID_N   = '1001064_N600_ev0.5_Normper_chip_median_PerChipScaleFalse'
DIR_N   = RETRIEVAL_BASE / RID_N
LABEL_N = '1001064'

# Na and Ca ARE in this run's model (Guidebook v4.2.5, same as RID_L/51459) -- use the
# FULL trace list so the CCF genuinely tests for them.
print('TRACE_SPECIES_BASE:', list(TRACE_SPECIES_BASE.values()))

with open(DIR_N / 'final_params_dict.pickle', 'rb') as f:
    best_fit_N = pickle.load(f)
print('Best-fit params (1001064):')
for k, v in best_fit_N.items():
    if np.ndim(v) == 0 and v is not None:
        print(f'  {k:25s} = {float(v):.5g}')

_CLOUD_SPECIES_N = list(_gb425.CLOUD_SPECIES_DEFAULT)   # MgSiO3 + Fe (Xuan+2024)
print('cloud_species:', _CLOUD_SPECIES_N)

# 008PM-EQ-CLD-MEDNORM-ATOMOPAC config: equilibrium + EddySed clouds + Na/Ca opacity,
# direct log_g, per-chip-median norm, GP -- matches
# tasting_retrieval_equa_chem_v6.1.5_piette_cloud_mednorm_atomopac.py as launched
# 2026-07-22. make_free_params_equil_chem_cloudy builds on make_free_params_equilibrium
# (v4.2.5, no log_Na/log_Ca) -> 24 params, same as 3708914.
constant_params_N = {'chemistry': 'equilibrium', 'cloud_species': _CLOUD_SPECIES_N}
free_params_N     = _gb425.make_free_params_equil_chem_cloudy(_CLOUD_SPECIES_N)
del free_params_N['log_M']
del free_params_N['log_R']
free_params_N['log_g'] = ({'type': 'gaussian', 'mu': 3.64, 'sigma': 0.20}, r'$\log g$')
free_params_N['log_a'] = ([-1.0, 1.0], r'$\log a_{\rm GP}$')
free_params_N['log_l'] = ([-3.0, 0.0], r'$\log l_{\rm GP}$')
assert 'log_Na' not in free_params_N and 'log_Ca' not in free_params_N, \
    'Guidebook v4.2.5 must NOT carry log_Na/log_Ca as free params'
print(f'free params: {len(free_params_N)} (expect 24)')

parameters_N = Parameters425(free_params_N, constant_params_N)
parameters_N(np.random.rand(parameters_N.ndim))
parameters_N.params.update(best_fit_N)

# Local per-chip-median helpers (self-contained -- same rationale as §16/17: never
# trust a warm kernel to hold a fixed version from a different section).
_K2166_N = np.array([
    [[2063.711, 2077.942], [2078.967, 2092.559], [2093.479, 2106.392]],
    [[2143.087, 2157.855], [2158.914, 2173.020], [2173.983, 2187.386]],
    [[2228.786, 2244.133], [2245.229, 2259.888], [2260.904, 2274.835]],
    [[2321.596, 2337.568], [2338.704, 2353.961], [2355.035, 2369.534]],
    [[2422.415, 2439.061], [2440.243, 2456.145], [2457.275, 2472.388]],
])

def _pcm_1d_N(wave, flux, err, K2166):
    flux = flux.copy(); err = err.copy()
    for order in range(5):
        for det in range(3):
            mask = (wave >= K2166[order, det, 0]) & (wave <= K2166[order, det, 1])
            if mask.sum() < 10: continue
            med = np.nanmedian(flux[mask])
            if med > 0: flux[mask] /= med; err[mask] /= med
    return flux, err

def _pcm_3d_N(wave3, flux3, err3, K2166):
    flux3 = flux3.copy(); err3 = err3.copy()
    for det in range(flux3.shape[0]):
        for order in range(flux3.shape[1]):
            fin = np.isfinite(flux3[det, order, :])
            if fin.sum() < 10:
                print(f'  [_pcm_3d_N] WARNING: det{det} ord{order}: only '
                      f'{fin.sum()} finite px — chip left un-normalised')
                continue
            med = np.nanmedian(flux3[det, order, fin])
            if med > 0: flux3[det, order, :] /= med; err3[det, order, :] /= med
    return flux3, err3

flux_N_N1, err_N_N1   = _pcm_1d_N(wave_N1, flux_N1, err_N1, _K2166_N)
flux_N_N2, err_N_N2   = _pcm_1d_N(wave_N2, flux_N2, err_N2, _K2166_N)
flux3_N_N1, err3_N_N1 = _pcm_3d_N(wave3_N1, flux3_N1, err3_N1, _K2166_N)
flux3_N_N2, err3_N_N2 = _pcm_3d_N(wave3_N2, flux3_N2, err3_N2, _K2166_N)

print(f'flux_N_N1 median: {np.nanmedian(flux_N_N1):.4f}  (should be ~1.0)')
print(f'flux_N_N2 median: {np.nanmedian(flux_N_N2):.4f}')

T1_N = Target425(wl=wave_N1, fl=flux_N_N1, err=err_N_N1, name='dh_tau_b_N_N1')
T2_N = Target425(wl=wave_N2, fl=flux_N_N2, err=err_N_N2, name='dh_tau_b_N_N2')

retrieval_N = Retrieval425(
    parameters         = parameters_N,
    N_live_points      = 600,
    evidence_tolerance = 0.5,
    targets            = [T1_N, T2_N],
    testing            = False,
    normalize_flux     = 'per_chip_median',
    per_chip_scaling   = False,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = False,
    cov_mode           = 'GP',
)
retrieval_N.parameters.params.update(best_fit_N)
print('Retrieval 1001064 (008PM-EQ-CLD-MEDNORM-ATOMOPAC, Na+Ca opacity, GP, v4.2.5) ready')


TRACE_SPECIES_BASE: ['H2O', '12CO', '13CO', 'CH4', 'FeH', 'HF', 'Na', 'Ca']
Best-fit params (1001064):
  rv_N1                     = 31.479
  rv_N2                     = 31.501
  vsini                     = 5.8828
  epsilon                   = 0.46292
  T_anchor                  = 1868.9
  dT_1                      = 536.7
  dT_2                      = 98.646
  dT_3                      = 107.45
  dT_4                      = 183.51
  dT_5                      = 165.06
  dT_6                      = 93.864
  dT_7                      = 458.68
  C_H                       = -0.33256
  C/O                       = 0.57891
  log_12CO_13CO             = 2.0988
  F_H                       = -0.94696
  log_X_MgSiO3              = -1.1423
  log_X_Fe                  = -1.4778
  fsed                      = 5.5024
  log_Kzz                   = 10.262
  sigma_lnorm               = 1.9598
  log_g                     = 3.6441
  log_a                     = 0.26717
  log_l                     = -2.2889


Loading equilibrium chemistry table (done once)...
Loading chemical equilibrium chemistry table from file '/net/lem/data2/pRT3_formatted/input_data/pre_calculated_chemistry/equilibrium_chemistry/equilibrium_chemistry.chemtable.petitRADTRANS.h5'... 

Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...


Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/H2O/1H2-16O/1H2-16O__POKAZATEL.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '12C-16O__HITEMP' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/CO/12C-16O/12C-16O__HITEMP.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

...

 Done.
 Loading line opacities of species '1H-19F' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/HF/1H-19F/1H-19F__Coxon-Hajig.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '23Na' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Na/23Na/23Na__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Loading line opacities of species '40Ca' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/lines/line_by_line/Ca/40Ca/40Ca__Kurucz.R1e+06_0.3-28.0mu.xsec.petitRADTRANS.h5'...

 Done.
 Successfully loaded all line opacities
 Loading CIA opacities for H2--H2 from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--H2/H2--H2-NatAbund/H2--H2-NatAbund__BoRi.R831_0.6-250mu.ciatable.petitRADTRANS.h5'... Done.
 Loading CIA opacities for H2--He from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/collision_induced_absorptions/H2--He/H2--He-NatAbund/H2--He-NatAbund__BoRi.DeltaWavenumber2_0.5-500mu.ciatable.petitRADTRANS.h5'... Done.
 Successfully loaded all CIA opacities
 Loading opacities of cloud species 'MgSiO3(s)_crystalline_000' from file '/net/lem/data2/pRT3_formatted/input_data/opacities/continuum/clouds/MgSiO3(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000/Mg-Si-O3-NatAbund(s)_crystalline_000__DHS.R39_0.1-250mu.cotable.petitRADTRANS.h5' (crystalline_000, using DHS scattering)... Done.
 Loading opacities of cloud species 'Fe(s)_crystalline_000' from file '/net/lem/data2/pRT3_formatted/

Retrieval 1001064 (008PM-EQ-CLD-MEDNORM-ATOMOPAC, Na+Ca opacity, GP, v4.2.5) ready


In [7]:
results_N = run_species_ccf_validation(
    retrieval          = retrieval_N,
    pRT_spectrum_class = pRT_spectrum425,
    best_fit_params    = best_fit_N,
    wave_model         = wave_model,
    obs_flux_N1        = flux3_N_N1,
    err_N1             = err3_N_N1,
    obs_wave_N1        = wave3_N1,
    obs_flux_N2        = flux3_N_N2,
    err_N2             = err3_N_N2,
    obs_wave_N2        = wave3_N2,
    TRACE_SPECIES      = TRACE_SPECIES_BASE,
    retrieval_dir      = DIR_N,
    retrieval_label    = LABEL_N,
    use_absolute_flux  = False,
)

import json as _json
_snr = {v['label']: dict(snr=round(float(v['snr']), 3),
                         peak_rv=float(v['peak_rv']),
                         template_fraction=float(v['template_fraction']))
        for v in results_N['ccf_results'].values()}
with open(DIR_N / 'validation_deregt_snr_summary.json', 'w') as f:
    _json.dump(_snr, f, indent=2)
print('SNR summary JSON saved to', DIR_N)

print()
print('=== Na / Ca CCF significance — 1001064 (cloud counterpart of 51459) ===')
for sp in ('Na', 'Ca'):
    found = False
    for prt_name, res in results_N['ccf_results'].items():
        if res['label'] == sp:
            found = True
            print(f"  {sp:3s}: peak CCF = {res['peak_val']:.3e}   "
                  f"peak_rv = {res['peak_rv']:+.1f} km/s   SNR = {res['snr']:.2f}   "
                  f"template_fraction = {res['template_fraction']:.1e}")
    if not found:
        print(f'  {sp:3s}: NOT in ccf_results (template negligible / not resolved in mass_fractions)')


  Planet RV: N1 = +31.479 km/s  N2 = +31.501 km/s
  Generating full model spectrum (Spectrum 1)...


  flux_all: [4.81e-01, 1.34e+00]
  Generating no-X templates...


    [H2O] template RMS = 1.03e-01
    [12CO] key resolved: '12C-16O' → '12C-16O__HITEMP'


    [12CO] template RMS = 5.93e-02


    [13CO] template RMS = 5.48e-03


...

  Saved: validation_deregt_HF.png


  Saved: validation_deregt_Na.png


  Saved: validation_deregt_Ca.png


  Saved: validation_deregt_totalCO.png


  Saved: validation_deregt_snr_summary.png
  CCF validation complete.
SNR summary JSON saved to /data2/peng/retrievals/1001064_N600_ev0.5_Normper_chip_median_PerChipScaleFalse

=== Na / Ca CCF significance — 1001064 (cloud counterpart of 51459) ===
  Na : peak CCF = 9.350e-01   peak_rv = -337.0 km/s   SNR = 4.21   template_fraction = 6.0e-03
  Ca : peak CCF = 2.773e-01   peak_rv = +540.0 km/s   SNR = 2.99   template_fraction = 3.0e-03
